In [1]:
!pip install catboost xgboost lightgbm torch pandas numpy scikit-learn tqdm matplotlib joblib

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter, defaultdict
from scipy import stats
from scipy.stats import skew, kurtosis, norm, shapiro

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (16, 8)
plt.rcParams["font.size"] = 11

# === Kaggle paths (EDIT nếu dataset khác tên) ===
DATA_DIR = "/kaggle/input/datasets/minhduylebao/dataeda"   # <- đổi nếu dataset name khác
FIGURES_DIR = "/kaggle/working/figures"
OUTPUT_DIR = "/kaggle/working/output"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

HORIZON = 28
TRAIN_LAST_D = None

SAVE = lambda name: plt.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight")

print("\n" + "=" * 90)
print(" " * 20 + "M5 FORECASTING - COMPREHENSIVE EDA")
print("=" * 90)
print("Sections: Statistics | Distribution | Trends | Seasonality | Events | Prices | SNAP")
print("          Data Quality | Hierarchy | Sparsity | Outliers | Autocorrelation | Features")
print("=" * 90)

# ============================================================================
# [1] LOAD DATA - Comprehensive metrics collection
# ============================================================================
print("\n[1] LOADING & ANALYZING DATA...")

CHUNK = 1_000_000
DTYPES = {
    "item_id":"category", "dept_id":"category", "cat_id":"category",
    "store_id":"category", "state_id":"category", "d":"category",
    "day_id":"uint16", "wm_yr_wk":"uint16", "sales":"float32",
    "sell_price":"float32", "wday":"uint8", "month":"uint8",
    "year":"uint16", "snap_CA":"uint8", "snap_TX":"uint8", "snap_WI":"uint8"
}

# === Initialize comprehensive counters ===
nrows_total = 0
sales_sum = sales_sq_sum = sales_cubed_sum = 0.0
sales_min, sales_max = float("inf"), float("-inf")
zero_count = positive_count = 0

# Store-level
store_sales = Counter()
store_zero = Counter()
store_items = defaultdict(set)
store_dates = defaultdict(set)

# Item-level
item_sales = Counter()
item_zero = Counter()
item_max = Counter()
item_min = Counter()
item_dates = defaultdict(set)
item_stores = defaultdict(set)
item_nonzero_days = Counter()

# Department & Category
dept_sales = Counter()
cat_sales = Counter()
state_sales = Counter()

# Temporal
dow_sales = Counter()
dow_cnt = Counter()
month_sales = Counter()
month_cnt = Counter()
year_sales = Counter()

# Events
event_sales_sum = event_sales_cnt = 0
noevent_sales_sum = noevent_sales_cnt = 0
event_type_1_cnt = Counter()
event_type_2_cnt = Counter()
event_name_1_cnt = Counter()
event_name_2_cnt = Counter()

# Price
price_sum = price_sq_sum = price_cnt = 0
price_min = float("inf")
price_max = float("-inf")
price_by_item = defaultdict(list)

# Missing per-column
missing_counts = Counter()

# Sampling
sampled_rows = []
SAMPLE_N = 250_000

LOG_EVERY = 1  # in log mỗi chunk

print(f"  -> DATA_DIR = {DATA_DIR}")
print("  -> Start loading train.csv ...")

for chunk_idx, chunk in enumerate(pd.read_csv(
    os.path.join(DATA_DIR, "train.csv"),
    chunksize=CHUNK, dtype=DTYPES, parse_dates=["date"]
)):
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"\n  [Chunk {chunk_idx+1}] read {len(chunk):,} rows")

    # ======= MISSING =======
    t0 = pd.Timestamp.now()
    miss = chunk.isna().sum()
    for k, v in miss.items():
        missing_counts[k] += int(v)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - missing count done ({(t1-t0).total_seconds():.2f}s)")

    # ======= SALES BASIC =======
    t0 = pd.Timestamp.now()
    n = len(chunk)
    nrows_total += n

    sales = chunk["sales"].values
    sales_sum += sales.sum()
    sales_sq_sum += (sales ** 2).sum()
    sales_cubed_sum += (sales ** 3).sum()
    sales_min = min(sales_min, sales.min())
    sales_max = max(sales_max, sales.max())

    zero_mask = sales == 0
    zero_count += zero_mask.sum()
    positive_count += (~zero_mask).sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - sales/basic done ({(t1-t0).total_seconds():.2f}s)")

    # ======= ITEM LEVEL =======
    t0 = pd.Timestamp.now()
    for item_id in chunk["id"].unique():
        item_chunk = chunk[chunk["id"] == item_id]
        item_s = item_chunk["sales"].sum()
        item_sales[item_id] += item_s
        item_zero[item_id] += int((item_chunk["sales"] == 0).sum())
        item_max[item_id] = max(item_max.get(item_id, 0), item_chunk["sales"].max())
        item_min[item_id] = min(item_min.get(item_id, float("inf")), item_chunk["sales"].min())
        item_nonzero_days[item_id] += int((item_chunk["sales"] > 0).sum())
        item_dates[item_id].update(item_chunk["date"].values)
        item_stores[item_id].update(item_chunk["store_id"].values)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - item level done ({(t1-t0).total_seconds():.2f}s)")

    # ======= STORE LEVEL =======
    t0 = pd.Timestamp.now()
    for store_id in chunk["store_id"].unique():
        store_chunk = chunk[chunk["store_id"] == store_id]
        store_sales[store_id] += store_chunk["sales"].sum()
        store_zero[store_id] += int((store_chunk["sales"] == 0).sum())
        store_items[store_id].update(store_chunk["id"].unique())
        store_dates[store_id].update(store_chunk["date"].values)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - store level done ({(t1-t0).total_seconds():.2f}s)")

    # ======= DEPT/CAT/STATE =======
    t0 = pd.Timestamp.now()
    for dept_id in chunk["dept_id"].unique():
        dept_chunk = chunk[chunk["dept_id"] == dept_id]
        dept_sales[dept_id] += dept_chunk["sales"].sum()

    for cat_id in chunk["cat_id"].unique():
        cat_chunk = chunk[chunk["cat_id"] == cat_id]
        cat_sales[cat_id] += cat_chunk["sales"].sum()

    for state_id in chunk["state_id"].unique():
        state_chunk = chunk[chunk["state_id"] == state_id]
        state_sales[state_id] += state_chunk["sales"].sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - dept/cat/state done ({(t1-t0).total_seconds():.2f}s)")

    # ======= TEMPORAL =======
    t0 = pd.Timestamp.now()
    for wday in chunk["wday"].unique():
        dow_chunk = chunk[chunk["wday"] == wday]
        dow_sales[wday] += dow_chunk["sales"].sum()
        dow_cnt[wday] += len(dow_chunk)

    for month in chunk["month"].unique():
        month_chunk = chunk[chunk["month"] == month]
        month_sales[month] += month_chunk["sales"].sum()
        month_cnt[month] += len(month_chunk)

    for year in chunk["year"].unique():
        year_chunk = chunk[chunk["year"] == year]
        year_sales[year] += year_chunk["sales"].sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - temporal done ({(t1-t0).total_seconds():.2f}s)")

    # ======= EVENTS =======
    t0 = pd.Timestamp.now()
    for et1 in chunk["event_type_1"].unique():
        if pd.notna(et1) and et1 != "None":
            et1_chunk = chunk[chunk["event_type_1"] == et1]
            event_type_1_cnt[et1] += len(et1_chunk)

    for et2 in chunk["event_type_2"].unique():
        if pd.notna(et2) and et2 != "None":
            et2_chunk = chunk[chunk["event_type_2"] == et2]
            event_type_2_cnt[et2] += len(et2_chunk)

    for en1 in chunk["event_name_1"].unique():
        if pd.notna(en1) and en1 != "None":
            en1_chunk = chunk[chunk["event_name_1"] == en1]
            event_name_1_cnt[en1] += len(en1_chunk)

    for en2 in chunk["event_name_2"].unique():
        if pd.notna(en2) and en2 != "None":
            en2_chunk = chunk[chunk["event_name_2"] == en2]
            event_name_2_cnt[en2] += len(en2_chunk)

    event_mask = (chunk["event_type_1"] != "None") | (chunk["event_type_2"] != "None")
    event_sales_sum += chunk.loc[event_mask, "sales"].sum()
    event_sales_cnt += event_mask.sum()
    noevent_sales_sum += chunk.loc[~event_mask, "sales"].sum()
    noevent_sales_cnt += (~event_mask).sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - events done ({(t1-t0).total_seconds():.2f}s)")

    # ======= PRICE =======
    t0 = pd.Timestamp.now()
    p = chunk["sell_price"].dropna().values
    price_sum += p.sum()
    price_sq_sum += (p ** 2).sum()
    price_cnt += len(p)
    if len(p) > 0:
        price_min = min(price_min, p.min())
        price_max = max(price_max, p.max())

    for item_id in chunk["id"].unique():
        item_prices = chunk[chunk["id"] == item_id]["sell_price"].dropna().values
        if len(item_prices) > 0:
            price_by_item[item_id].extend(item_prices)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - price done ({(t1-t0).total_seconds():.2f}s)")

    # ======= SAMPLING =======
    t0 = pd.Timestamp.now()
    if len(sampled_rows) < SAMPLE_N:
        idx = np.random.choice(n, size=min(n, SAMPLE_N - len(sampled_rows)), replace=False)
        sampled_rows.append(chunk.iloc[idx])
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - sampling done ({(t1-t0).total_seconds():.2f}s)")

    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    > total so far: rows={nrows_total:,} | items={len(item_sales):,} | zeros={zero_count:,}")

print(f"\n  ✓ Done train.csv: {nrows_total:,} rows loaded in {chunk_idx+1} chunks")

print("  -> Start loading val.csv ...")
val_shape = sum(1 for _ in open(os.path.join(DATA_DIR, "val.csv"))) - 1
val = pd.read_csv(os.path.join(DATA_DIR, "val.csv"), dtype=DTYPES, parse_dates=["date"])
print(f"  ✓ Done val.csv: {val_shape:,} rows")

print("  -> Start loading test.csv ...")
test_shape = sum(1 for _ in open(os.path.join(DATA_DIR, "test.csv"))) - 1
print(f"  ✓ Done test.csv: {test_shape:,} rows")

train_sample = pd.concat(sampled_rows, ignore_index=True)
n_items = len(item_sales)
n_stores = len(store_sales)
n_depts = len(dept_sales)
n_cats = len(cat_sales)
n_states = len(state_sales)

stores_sorted = sorted(store_sales.keys())
items_sorted = sorted(item_sales.keys())
depts_sorted = sorted(dept_sales.keys())
cats_sorted = sorted(cat_sales.keys())

print(f"  ✓ Items: {n_items:,} | Stores: {n_stores} | Depts: {n_depts} | Categories: {n_cats} | States: {n_states}")
print(f"  ✓ Train: {nrows_total:,} | Val: {val_shape:,} | Test: {test_shape:,}")
print(f"  ✓ Sample: {len(train_sample):,} rows")

# ============================================================================
# [2] BASIC STATISTICS & DATA QUALITY
# ============================================================================
print("\n[2] STATISTICS & DATA QUALITY...")

sales_mean = sales_sum / nrows_total if nrows_total > 0 else 0
sales_var = (sales_sq_sum / nrows_total - sales_mean ** 2) if nrows_total > 0 else 0
sales_std = np.sqrt(max(sales_var, 0))
sales_skew_est = sales_cubed_sum / nrows_total / (sales_std ** 3) if sales_std > 0 else 0
zero_pct = zero_count / nrows_total * 100 if nrows_total > 0 else 0
sparsity = zero_pct / 100

price_mean = price_sum / price_cnt if price_cnt > 0 else 0
price_var = price_sq_sum / price_cnt - price_mean ** 2 if price_cnt > 0 else 0
price_std = np.sqrt(max(price_var, 0))

# Event impact
base_avg = noevent_sales_sum / max(noevent_sales_cnt, 1) if noevent_sales_cnt > 0 else 0
event_avg = event_sales_sum / max(event_sales_cnt, 1) if event_sales_cnt > 0 else 0
event_lift = (event_avg / base_avg - 1) * 100 if base_avg > 0 else 0

# Item metrics
items_zero_pct = np.array([item_zero[i] / max(item_nonzero_days[i] + item_zero[i], 1) * 100 
                           for i in items_sorted])
items_velocity = np.array([item_nonzero_days[i] / max(item_zero[i] + item_nonzero_days[i], 1) 
                           for i in items_sorted])

print(f"\n  Sales Statistics:")
print(f"    Mean:           {sales_mean:12.4f}")
print(f"    Std Dev:        {sales_std:12.4f}")
print(f"    Min:            {sales_min:12.4f}")
print(f"    Max:            {sales_max:12.4f}")
print(f"    Skewness (est): {sales_skew_est:12.4f}")
print(f"    CV (Coeff Var): {sales_std/sales_mean:12.4f}" if sales_mean > 0 else "")

print(f"\n  Sparsity & Zero Sales:")
print(f"    Zero counts:    {zero_count:13,} ({zero_pct:6.2f}%)")
print(f"    Positive:       {positive_count:13,} ({100-zero_pct:6.2f}%)")

print(f"\n  Pricing:")
print(f"    Mean:           {price_mean:12.4f}")
print(f"    Std Dev:        {price_std:12.4f}")
print(f"    Min:            {price_min:12.4f}")
print(f"    Max:            {price_max:12.4f}")
print(f"    Records:        {price_cnt:13,}")

print(f"\n  Events Impact:")
print(f"    Avg w/o event:  {base_avg:12.4f}")
print(f"    Avg w/ event:   {event_avg:12.4f}")
print(f"    Lift:           {event_lift:12.2f}%")

print(f"\n  Items Characteristics:")
print(f"    Items (total):  {len(item_sales):13,}")
print(f"    Items w/ sales: {sum(1 for v in item_nonzero_days.values() if v > 0):13,}")
print(f"    Avg stores/item:{np.mean([len(item_stores[i]) for i in items_sorted]):13.1f}")
print(f"    Avg zero % (item): {items_zero_pct.mean():12.2f}%")
print(f"    Velocity (avg): {items_velocity.mean():12.4f}")

# ============================================================================
# [3] DISTRIBUTION ANALYSIS
# ============================================================================
print("\n[3] SALES DISTRIBUTION...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 3.1: Log distribution
ax = axes[0, 0]
s_nz = train_sample[train_sample["sales"] > 0]["sales"]
ax.hist(np.log1p(s_nz), bins=100, edgecolor="white", linewidth=0.3, alpha=0.7, color="steelblue")
ax.axvline(np.log1p(s_nz.median()), color="red", ls="--", lw=2, label=f"Median={np.log1p(s_nz.median()):.2f}")
ax.axvline(np.log1p(s_nz.mean()), color="orange", ls="--", lw=2, label=f"Mean={np.log1p(s_nz.mean()):.2f}")
ax.set_xlabel("log(1+sales)"); ax.set_ylabel("Frequency")
ax.set_title("Sales Distribution (log, non-zero)", fontweight="bold", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 3.2: Raw distribution (first 50 units)
ax = axes[0, 1]
ax.hist(s_nz[s_nz <= 50], bins=50, edgecolor="white", linewidth=0.5, alpha=0.7, color="coral")
ax.set_xlabel("Sales (units)"); ax.set_ylabel("Frequency")
ax.set_title("Sales Distribution (0-50 units)", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3)

# 3.3: Q-Q plot for normality check
ax = axes[0, 2]
log_sales = np.log1p(s_nz.values)
stats.probplot(log_sales, dist="norm", plot=ax)
ax.set_title(f"Q-Q Plot (log sales)\nSkew={skew(log_sales):.2f}", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3)

# 3.4: Store-wise zero percentage
ax = axes[1, 0]
n_per_store = nrows_total // n_stores
zero_pcts = [store_zero[s] / n_per_store * 100 for s in stores_sorted]
bars = ax.bar(range(len(stores_sorted)), zero_pcts, color="lightcoral", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted)))
ax.set_xticklabels(stores_sorted, rotation=0)
ax.set_ylabel("Zero Sales %"); ax.set_title("Zero Sales % by Store", fontweight="bold", fontsize=12)
ax.axhline(np.mean(zero_pcts), color="red", ls="--", lw=1, label=f"Avg: {np.mean(zero_pcts):.1f}%")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

# 3.5: Mean sales by store
ax = axes[1, 1]
means_store = [store_sales[s] / (nrows_total // n_stores) for s in stores_sorted]
bars = ax.bar(range(len(stores_sorted)), means_store, color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted)))
ax.set_xticklabels(stores_sorted, rotation=0)
ax.set_ylabel("Mean Daily Sales"); ax.set_title("Avg Daily Sales by Store", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3, axis="y")

# 3.6: Mean sales by category
ax = axes[1, 2]
cat_avg = {c: cat_sales[c] / max(1, nrows_total // n_cats) for c in cats_sorted}
cats_sorted_by_sales = sorted(cat_avg.keys(), key=lambda c: cat_avg[c], reverse=True)
cat_vals = [cat_avg[c] for c in cats_sorted_by_sales]
bars = ax.bar(range(len(cat_vals)), cat_vals, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(cats_sorted_by_sales)))
ax.set_xticklabels(cats_sorted_by_sales, rotation=45, ha="right")
ax.set_ylabel("Mean Daily Sales"); ax.set_title("Avg Daily Sales by Category", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("01_distribution.png")
plt.close()
print("  ✓ Saved: 01_distribution.png")

# ============================================================================
# [4] TREND ANALYSIS
# ============================================================================
print("\n[4] TREND ANALYSIS...")

train_sample["date"] = pd.to_datetime(train_sample["date"])
daily = train_sample.groupby("date")["sales"].agg(["sum", "mean", "std", "count"]).reset_index()
daily.columns = ["date", "total_sales", "mean_sales", "std_sales", "n_records"]

val_daily = val.groupby("date")["sales"].agg(["sum", "mean"]).reset_index()
val_daily.columns = ["date", "total_sales", "mean_sales"]

fig, axes = plt.subplots(3, 2, figsize=(20, 15))

# 4.1: Total sales over time with MA
ax = axes[0, 0]
ax.plot(daily["date"], daily["total_sales"], lw=0.5, color="steelblue", alpha=0.5, label="Daily Sales")
ax.plot(daily["date"], daily["total_sales"].rolling(7, center=True).mean(), 
        color="darkblue", lw=1.5, label="7-day MA")
ax.plot(daily["date"], daily["total_sales"].rolling(30, center=True).mean(), 
        color="red", lw=1.5, label="30-day MA")
ax.plot(val_daily["date"], val_daily["total_sales"], lw=0.6, color="green", alpha=0.6, label="Val (2016)")
ax.axvline(pd.Timestamp("2016-04-25"), color="black", ls="--", lw=1, alpha=0.5)
ax.set_ylabel("Total Sales"); ax.set_title("Daily Sales Trend with Moving Averages", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.2: Sales volatility (rolling std)
ax = axes[0, 1]
ax.plot(daily["date"], daily["std_sales"], lw=0.7, color="orange", alpha=0.7)
ax.plot(daily["date"], daily["std_sales"].rolling(30).mean(), color="red", lw=2, label="30-day MA")
ax.set_ylabel("Std Dev of Sales"); ax.set_title("Sales Volatility Over Time", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.3: Year-over-year
ax = axes[1, 0]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
for year_idx, year in enumerate([2011, 2012, 2013, 2014, 2015, 2016]):
    y = daily[daily["date"].dt.year == year].copy()
    if not y.empty:
        y["doy"] = y["date"].dt.dayofyear
        ax.plot(y["doy"], y["total_sales"], lw=1.2, alpha=0.8, label=str(year), color=colors[year_idx])
ax.set_xlabel("Day of Year"); ax.set_ylabel("Total Sales")
ax.set_title("Year-over-Year Comparison", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.4: Growth rate
ax = axes[1, 1]
daily_sorted = daily.sort_values("date").reset_index(drop=True)
growth = daily_sorted["total_sales"].pct_change().rolling(7).mean() * 100
ax.plot(daily_sorted["date"], growth, lw=0.8, color="purple")
ax.axhline(0, color="black", ls="-", lw=0.5)
ax.fill_between(daily_sorted["date"], growth, 0, where=(growth >= 0), alpha=0.3, color="green")
ax.fill_between(daily_sorted["date"], growth, 0, where=(growth < 0), alpha=0.3, color="red")
ax.set_ylabel("Growth Rate (%)"); ax.set_title("7-day Rolling Growth Rate", fontweight="bold")
ax.grid(alpha=0.3)

# 4.5: Monthly totals
ax = axes[2, 0]
monthly = train_sample.copy()
monthly["ym"] = monthly["date"].dt.to_period("M")
monthly_sales = monthly.groupby("ym")["sales"].sum()
ax.bar(range(len(monthly_sales)), monthly_sales.values, color="steelblue", edgecolor="black", alpha=0.7)
ax.set_ylabel("Total Sales"); ax.set_title("Monthly Sales Aggregates", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 4.6: State-wise trends
ax = axes[2, 1]
for state in sorted(state_sales.keys()):
    state_data = train_sample[train_sample["state_id"] == state]
    state_daily = state_data.groupby("date")["sales"].sum().rolling(7).mean()
    ax.plot(state_daily.index, state_daily.values, lw=1, label=state, alpha=0.8)
ax.set_ylabel("7-day MA Sales"); ax.set_title("State-wise Sales Trends", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
SAVE("02_trends.png")
plt.close()
print("  ✓ Saved: 02_trends.png")

# ============================================================================
# [5] SEASONALITY ANALYSIS
# ============================================================================
print("\n[5] SEASONALITY...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 5.1: Day of week
ax = axes[0, 0]
dow_labels = ["Sat", "Sun", "Mon", "Tue", "Wed", "Thu", "Fri"]
dow_n = [dow_cnt.get(d, 1) for d in range(1, 8)]
dow_vals = [dow_sales.get(d, 0) / max(n, 1) for d, n in zip(range(1, 8), dow_n)]
colors_dow = ["salmon" if d in [1, 2] else "skyblue" for d in range(1, 8)]
bars = ax.bar(dow_labels, dow_vals, color=colors_dow, edgecolor="black", linewidth=1)
ax.set_ylabel("Avg Sales"); ax.set_title("Day of Week Effect", fontweight="bold")
for bar, v in zip(bars, dow_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(dow_vals)*0.02, 
            f"{v:.0f}", ha="center", fontsize=10, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.2: Month
ax = axes[0, 1]
month_vals = [month_sales.get(m, 0) for m in range(1, 13)]
colors_m = ["#FFB3BA" if m in [10, 11, 12] else "#BAE1FF" for m in range(1, 13)]
labels_m = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bars = ax.bar(labels_m, month_vals, color=colors_m, edgecolor="black", linewidth=1)
ax.set_ylabel("Total Sales"); ax.set_title("Seasonal Pattern by Month", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.3: Weekend vs Weekday
ax = axes[0, 2]
w_avg = sum(dow_sales.get(d, 0) for d in [1, 2]) / max(sum(dow_cnt.get(d, 1) for d in [1, 2]), 1)
d_avg = sum(dow_sales.get(d, 0) for d in [3, 4, 5, 6, 7]) / max(sum(dow_cnt.get(d, 1) for d in [3, 4, 5, 6, 7]), 1)
bars = ax.bar(["Weekend", "Weekday"], [w_avg, d_avg], color=["salmon", "skyblue"], 
              edgecolor="black", width=0.5, linewidth=1)
ax.set_ylabel("Avg Daily Sales"); ax.set_title("Weekend vs Weekday", fontweight="bold")
for bar, v in zip(bars, [w_avg, d_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max([w_avg, d_avg])*0.02, 
            f"{v:.0f}", ha="center", fontsize=11, fontweight="bold")
wk_ratio = (w_avg / d_avg - 1) * 100 if d_avg > 0 else 0
ax.text(0.5, max([w_avg, d_avg]) * 0.5, f"Diff: {wk_ratio:+.1f}%", ha="center", fontsize=12, 
        bbox=dict(boxstyle="round", facecolor="yellow", alpha=0.3), fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.4: Event impact
ax = axes[1, 0]
bars = ax.bar(["No Event", "Has Event"], [base_avg, event_avg], 
              color=["lightgray", "goldenrod"], edgecolor="black", width=0.5, linewidth=1)
ax.set_ylabel("Avg Daily Sales"); ax.set_title(f"Event Impact (Lift: {event_lift:+.1f}%)", fontweight="bold")
for bar, v in zip(bars, [base_avg, event_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max([base_avg, event_avg])*0.02, 
            f"{v:.0f}", ha="center", fontsize=11, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.5: Autocorrelation
ax = axes[1, 1]
daily_sales_for_acf = daily.sort_values("date")["total_sales"].values
lags = np.arange(1, 61)
acf_values = [np.corrcoef(daily_sales_for_acf[:-lag], daily_sales_for_acf[lag:])[0, 1] for lag in lags]
ax.plot(lags, acf_values, lw=1.5, color="steelblue", marker="o", markersize=3)
ax.axhline(0, color="black", ls="-", lw=0.5)
ax.axhline(1.96/np.sqrt(len(daily_sales_for_acf)), color="red", ls="--", lw=1, label="95% CI")
ax.axhline(-1.96/np.sqrt(len(daily_sales_for_acf)), color="red", ls="--", lw=1)
ax.set_xlabel("Lag (days)"); ax.set_ylabel("Autocorrelation"); ax.set_title("Autocorrelation (ACF)", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 5.6: Quarterly pattern
ax = axes[1, 2]
daily["quarter"] = daily["date"].dt.quarter
quarterly = daily.groupby("quarter")["total_sales"].sum()
quarters = ["Q1", "Q2", "Q3", "Q4"]
bars = ax.bar(quarters[:len(quarterly)], quarterly.values, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_ylabel("Total Sales"); ax.set_title("Quarterly Sales Pattern", fontweight="bold")
for bar, v in zip(bars, quarterly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + quarterly.max()*0.02, 
            f"{v/1e6:.1f}M", ha="center", fontsize=10, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("03_seasonality.png")
plt.close()
print("  ✓ Saved: 03_seasonality.png")

# ============================================================================
# [6] EVENT & SNAP ANALYSIS
# ============================================================================
print("\n[6] EVENTS & SNAP...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 6.1: Event types
ax = axes[0, 0]
et_counts = sorted(event_type_1_cnt.items(), key=lambda x: x[1], reverse=True)[:10]
if et_counts:
    names, counts = zip(*et_counts)
    ax.barh(names, counts, color="steelblue", edgecolor="black", linewidth=1)
    ax.set_xlabel("Frequency"); ax.set_title("Top 10 Event Types 1", fontweight="bold")
    ax.grid(alpha=0.3, axis="x")

# 6.2: Event names
ax = axes[0, 1]
en_counts = sorted(event_name_1_cnt.items(), key=lambda x: x[1], reverse=True)[:10]
if en_counts:
    names, counts = zip(*en_counts)
    ax.barh(names, counts, color="coral", edgecolor="black", linewidth=1)
    ax.set_xlabel("Frequency"); ax.set_title("Top 10 Event Names 1", fontweight="bold")
    ax.grid(alpha=0.3, axis="x")

# 6.3: SNAP impact by state
ax = axes[1, 0]
snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
snap_data = {}
for state, snap_col in snap_cols.items():
    state_df = train_sample[train_sample["state_id"] == state]
    snap_0 = state_df[state_df[snap_col] == 0]["sales"].mean()
    snap_1 = state_df[state_df[snap_col] == 1]["sales"].mean()
    snap_data[state] = (snap_0, snap_1)

x = np.arange(len(snap_data))
width = 0.35
bars1 = ax.bar(x - width/2, [v[0] for v in snap_data.values()], width, label="No SNAP", 
               color="lightcoral", edgecolor="black", linewidth=1)
bars2 = ax.bar(x + width/2, [v[1] for v in snap_data.values()], width, label="SNAP", 
               color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_ylabel("Avg Sales"); ax.set_title("SNAP Program Impact by State", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(snap_data.keys())
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

for i, (state, (s0, s1)) in enumerate(snap_data.items()):
    lift = (s1 / s0 - 1) * 100 if s0 > 0 else 0
    ax.text(i, max(s0, s1) * 1.05, f"{lift:+.1f}%", ha="center", fontsize=10, fontweight="bold")

# 6.4: Event frequency by month
ax = axes[1, 1]
event_monthly = train_sample[train_sample["event_type_1"] != "None"].copy()
if len(event_monthly) > 0:
    event_monthly["month_name"] = pd.to_datetime(event_monthly["date"]).dt.strftime("%b")
    event_cnt = event_monthly.groupby("month_name").size()
    months_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    event_cnt = event_cnt.reindex([m for m in months_order if m in event_cnt.index])
    ax.bar(range(len(event_cnt)), event_cnt.values, color="goldenrod", edgecolor="black", linewidth=1)
    ax.set_xticks(range(len(event_cnt))); ax.set_xticklabels(event_cnt.index, rotation=45)
    ax.set_ylabel("Event Count"); ax.set_title("Event Frequency by Month", fontweight="bold")
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("04_events_snap.png")
plt.close()
print("  ✓ Saved: 04_events_snap.png")

# ============================================================================
# [7] PRICE ANALYSIS
# ============================================================================
print("\n[7] PRICE ANALYSIS...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 7.1: Price distribution
ax = axes[0, 0]
prices = train_sample["sell_price"].dropna()
ax.hist(prices, bins=100, edgecolor="white", alpha=0.7, color="mediumblue")
ax.axvline(prices.mean(), color="red", ls="--", lw=2, label=f"Mean: {prices.mean():.2f}")
ax.axvline(prices.median(), color="orange", ls="--", lw=2, label=f"Median: {prices.median():.2f}")
ax.set_xlabel("Price ($)"); ax.set_ylabel("Frequency")
ax.set_title("Sell Price Distribution", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 7.2: Price by category
ax = axes[0, 1]
cat_price = train_sample.groupby("cat_id")["sell_price"].agg(["mean", "std"]).sort_values("mean", ascending=False)
x = np.arange(len(cat_price))
ax.bar(x, cat_price["mean"], yerr=cat_price["std"], color="steelblue", edgecolor="black", 
       linewidth=1, capsize=5, alpha=0.7, error_kw={"linewidth": 2})
ax.set_xticks(x); ax.set_xticklabels(cat_price.index, rotation=0)
ax.set_ylabel("Price ($)"); ax.set_title("Mean Price by Category (with std)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 7.3: Price volatility (top items)
ax = axes[0, 2]
item_price_vol = {}
for item_id in train_sample["id"].unique():
    prices_item = train_sample[train_sample["id"] == item_id]["sell_price"].dropna()
    if len(prices_item) > 0:
        item_price_vol[item_id] = prices_item.std()
top_vol = sorted(item_price_vol.items(), key=lambda x: x[1], reverse=True)[:15]
names, vols = zip(*top_vol)
ax.barh(range(len(names)), vols, color="darkorange", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("Price Std Dev ($)"); ax.set_title("Top 15 Items by Price Volatility", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 7.4: Price-Sales correlation
ax = axes[1, 0]
sample_for_corr = train_sample[["sales", "sell_price"]].dropna()
corr_coef = sample_for_corr["sales"].corr(sample_for_corr["sell_price"])
ax.scatter(sample_for_corr["sell_price"], sample_for_corr["sales"], alpha=0.3, s=10, color="steelblue")
ax.set_xlabel("Price ($)"); ax.set_ylabel("Sales (units)")
ax.set_title(f"Price-Sales Relationship (corr: {corr_coef:.4f})", fontweight="bold")
ax.grid(alpha=0.3)

# 7.5: Price changes over time
ax = axes[1, 1]
for item in list(train_sample["id"].unique())[:5]:  # Top 5 items
    item_data = train_sample[train_sample["id"] == item].sort_values("date")
    if len(item_data) > 100:
        prices_rolling = item_data.groupby(item_data["date"].dt.to_period("M"))["sell_price"].mean()
        ax.plot(range(len(prices_rolling)), prices_rolling.values, lw=1.5, label=item, alpha=0.7)
ax.set_xlabel("Month"); ax.set_ylabel("Avg Price ($)"); ax.set_title("Price Trends (Top 5 Items)", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 7.6: Price by store
ax = axes[1, 2]
store_prices = {}
for store in stores_sorted:
    store_df = train_sample[train_sample["store_id"] == store]
    store_prices[store] = store_df["sell_price"].mean()
ax.bar(range(len(store_prices)), list(store_prices.values()), color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(store_prices))); ax.set_xticklabels(list(store_prices.keys()))
ax.set_ylabel("Mean Price ($)"); ax.set_title("Avg Price by Store", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("05_prices.png")
plt.close()
print("  ✓ Saved: 05_prices.png")

# ============================================================================
# [8] SPARSITY & ITEM ANALYSIS
# ============================================================================
print("\n[8] SPARSITY & ITEM ANALYSIS...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 8.1: Item zero-sale distribution
ax = axes[0, 0]
item_zero_pcts = np.array([item_zero[i] / max(item_zero[i] + item_nonzero_days[i], 1) * 100 
                           for i in items_sorted])
ax.hist(item_zero_pcts, bins=50, edgecolor="white", alpha=0.7, color="coral")
ax.axvline(item_zero_pcts.mean(), color="red", ls="--", lw=2, label=f"Mean: {item_zero_pcts.mean():.1f}%")
ax.set_xlabel("Zero Sales %"); ax.set_ylabel("Frequency")
ax.set_title("Distribution of Item Zero-Sale Rates", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.2: Items by sparsity category
ax = axes[0, 1]
sparsity_bins = [0, 25, 50, 75, 100]
sparsity_labels = ["<25%", "25-50%", "50-75%", ">75%"]
item_sparsity_cats = pd.cut(item_zero_pcts, bins=sparsity_bins, labels=sparsity_labels, include_lowest=True)
sparsity_counts = item_sparsity_cats.value_counts().sort_index()
colors_sparsity = ["green", "yellow", "orange", "red"]
bars = ax.bar(range(len(sparsity_counts)), sparsity_counts.values, color=colors_sparsity, 
              edgecolor="black", linewidth=1)
ax.set_xticks(range(len(sparsity_counts))); ax.set_xticklabels(sparsity_counts.index)
ax.set_ylabel("Item Count"); ax.set_title("Items by Sparsity Category", fontweight="bold")
for bar, v in zip(bars, sparsity_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + sparsity_counts.max()*0.02, 
            f"{v}\n({v/len(items_sorted)*100:.1f}%)", ha="center", fontsize=9)
ax.grid(alpha=0.3, axis="y")

# 8.3: Item velocity (active days ratio)
ax = axes[0, 2]
item_velocities = np.array([item_nonzero_days[i] / max(item_zero[i] + item_nonzero_days[i], 1) 
                            for i in items_sorted])
ax.hist(item_velocities, bins=50, edgecolor="white", alpha=0.7, color="steelblue")
ax.axvline(item_velocities.mean(), color="red", ls="--", lw=2, label=f"Mean: {item_velocities.mean():.2f}")
ax.set_xlabel("Velocity (fraction of non-zero days)"); ax.set_ylabel("Frequency")
ax.set_title("Item Velocity Distribution", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.4: Items per store distribution
ax = axes[1, 0]
items_per_store = np.array([len(store_items[s]) for s in stores_sorted])
ax.bar(range(len(stores_sorted)), items_per_store, color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted))); ax.set_xticklabels(stores_sorted)
ax.set_ylabel("Number of Items"); ax.set_title("Items per Store", fontweight="bold")
ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(items_per_store):
    ax.text(i, v + items_per_store.max()*0.02, str(int(v)), ha="center", fontsize=9)

# 8.5: Store coverage by item
ax = axes[1, 1]
stores_per_item = np.array([len(item_stores[i]) for i in items_sorted])
ax.hist(stores_per_item, bins=n_stores+1, edgecolor="white", alpha=0.7, color="mediumpurple")
ax.axvline(stores_per_item.mean(), color="red", ls="--", lw=2, label=f"Mean: {stores_per_item.mean():.1f}")
ax.set_xlabel("Number of Stores"); ax.set_ylabel("Item Count")
ax.set_title("Store Coverage by Item", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.6: Top items by sales
ax = axes[1, 2]
top_items = sorted(item_sales.items(), key=lambda x: x[1], reverse=True)[:15]
item_names, item_sales_vals = zip(*top_items)
ax.barh(range(len(item_names)), item_sales_vals, color="darkorange", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(item_names))); ax.set_yticklabels(item_names)
ax.set_xlabel("Total Sales"); ax.set_title("Top 15 Items by Sales", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
SAVE("06_sparsity.png")
plt.close()
print("  ✓ Saved: 06_sparsity.png")

# ============================================================================
# [9] OUTLIER & ANOMALY DETECTION
# ============================================================================
print("\n[9] OUTLIERS & ANOMALIES...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 9.1: Boxplot of sales (log scale)
ax = axes[0, 0]
sales_nz_log = np.log1p(train_sample[train_sample["sales"] > 0]["sales"])
ax.boxplot([sales_nz_log], labels=["Sales (log)"], patch_artist=True,
           boxprops=dict(facecolor="lightblue", edgecolor="black"),
           whiskerprops=dict(color="black"), capprops=dict(color="black"))
ax.set_ylabel("log(1+sales)"); ax.set_title("Sales Outliers (Boxplot)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 9.2: Daily anomalies (z-score)
ax = axes[0, 1]
daily_sales_clean = daily.sort_values("date")["total_sales"].values
z_scores = np.abs((daily_sales_clean - np.mean(daily_sales_clean)) / np.std(daily_sales_clean))
daily_copy = daily.sort_values("date").copy()
daily_copy["z_score"] = z_scores
daily_copy["anomaly"] = z_scores > 2.5

ax.plot(daily_copy["date"], daily_copy["total_sales"], lw=0.6, color="steelblue", alpha=0.5, label="Daily Sales")
ax.scatter(daily_copy[daily_copy["anomaly"]]["date"], 
          daily_copy[daily_copy["anomaly"]]["total_sales"], 
          color="red", s=50, label="Anomalies (z>2.5)", zorder=5)
ax.set_ylabel("Total Sales"); ax.set_title("Daily Sales Anomalies", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 9.3: Item max sales (top outliers)
ax = axes[1, 0]
item_max_vals = [(i, item_max[i]) for i in items_sorted]
item_max_vals.sort(key=lambda x: x[1], reverse=True)
top_max = item_max_vals[:15]
names, maxs = zip(*top_max)
ax.barh(range(len(names)), maxs, color="crimson", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("Max Daily Sales"); ax.set_title("Top 15 Items by Max Daily Sales", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 9.4: Price outliers
ax = axes[1, 1]
price_data = train_sample["sell_price"].dropna()
if len(price_data) > 0:
    Q1, Q3 = price_data.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    price_normal = price_data[(price_data >= lower_bound) & (price_data <= upper_bound)]
    price_outliers = price_data[(price_data < lower_bound) | (price_data > upper_bound)]
    
    ax.scatter(range(len(price_normal)), price_normal.values, alpha=0.3, s=5, color="steelblue", label="Normal")
    ax.scatter([i + len(price_normal) for i in range(len(price_outliers))], price_outliers.values, 
              alpha=0.6, s=20, color="red", label="Outliers")
    ax.axhline(lower_bound, color="orange", ls="--", lw=1, label="Lower bound")
    ax.axhline(upper_bound, color="orange", ls="--", lw=1, label="Upper bound")
    ax.set_xlabel("Record"); ax.set_ylabel("Price ($)"); ax.set_title("Price Outliers (IQR method)", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
SAVE("07_outliers.png")
plt.close()
print("  ✓ Saved: 07_outliers.png")

# ============================================================================
# [10] HIERARCHY & ROLLUP ANALYSIS
# ============================================================================
print("\n[10] HIERARCHY ANALYSIS...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 10.1: State sales composition
ax = axes[0, 0]
state_sales_sorted = sorted(state_sales.items(), key=lambda x: x[1], reverse=True)
states, state_vals = zip(*state_sales_sorted)
colors_state = ["#FF9999", "#66B2FF", "#99FF99"]
ax.pie(state_vals, labels=states, autopct="%1.1f%%", colors=colors_state, startangle=90)
ax.set_title("Sales by State", fontweight="bold", fontsize=12)

# 10.2: Store sales within states
ax = axes[0, 1]
x = np.arange(len(stores_sorted))
width = 0.6
store_colors = {"CA": "red", "TX": "blue", "WI": "green"}
colors = [store_colors.get(s[:2], "gray") for s in stores_sorted]
stores_vals = [store_sales[s] for s in stores_sorted]
ax.bar(x, stores_vals, width, color=colors, edgecolor="black", linewidth=1, alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(stores_sorted)
ax.set_ylabel("Total Sales"); ax.set_title("Sales by Store (colored by state)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 10.3: Category sales composition
ax = axes[1, 0]
cat_sales_sorted = sorted(cat_sales.items(), key=lambda x: x[1], reverse=True)
cats, cat_vals = zip(*cat_sales_sorted)
ax.pie(cat_vals, labels=cats, autopct="%1.1f%%", startangle=45)
ax.set_title("Sales by Category", fontweight="bold", fontsize=12)

# 10.4: Department in categories
ax = axes[1, 1]
dept_cat_map = {}
for dept in depts_sorted:
    if dept.startswith("D"):
        cat = dept[0]
        if cat not in dept_cat_map:
            dept_cat_map[cat] = []
        dept_cat_map[cat].append(dept_sales[dept])

cat_labels = sorted(dept_cat_map.keys())
x_pos = np.arange(len(cat_labels))
dept_sales_by_cat = [sum(dept_cat_map[c]) for c in cat_labels]
ax.bar(x_pos, dept_sales_by_cat, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(x_pos); ax.set_xticklabels(cat_labels)
ax.set_ylabel("Total Sales"); ax.set_title("Sales by Category (Dept Rollup)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("08_hierarchy.png")
plt.close()
print("  ✓ Saved: 08_hierarchy.png")

# ============================================================================
# [11] DATA QUALITY & COMPLETENESS
# ============================================================================
print("\n[11] DATA QUALITY...")

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 11.1: Date coverage
ax = axes[0, 0]
date_range = (train_sample["date"].max() - train_sample["date"].min()).days
expected_rows = date_range * n_items * n_stores
actual_rows = len(train_sample)
coverage = actual_rows / expected_rows * 100 if expected_rows > 0 else 0
ax.text(0.5, 0.7, f"Date Coverage: {coverage:.1f}%", ha="center", fontsize=14, fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8))
ax.text(0.5, 0.5, f"Date Range: {date_range} days", ha="center", fontsize=12)
ax.text(0.5, 0.35, f"Expected rows: {expected_rows:,}\nActual rows: {actual_rows:,}", 
        ha="center", fontsize=11)
ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# 11.2: Missing values per column
ax = axes[0, 1]
missing_cols = dict(sorted(missing_counts.items(), key=lambda x: x[1], reverse=True))
missing_pcts = {k: v / nrows_total * 100 for k, v in missing_cols.items()}
ax.bar(missing_pcts.keys(), missing_pcts.values(), color="coral", edgecolor="black", linewidth=1)
ax.set_ylabel("Missing %"); ax.set_title("Missing Data by Column", fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.3, axis="y")

# 11.3: Data types memory usage (rough estimate)
ax = axes[1, 0]
dtypes_memory = {
    "sales (float32)": nrows_total * 4 / 1e6,
    "sell_price (float32)": price_cnt * 4 / 1e6,
    "store_id (category)": n_stores * 10 / 1e6,
    "item_id (category)": n_items * 10 / 1e6,
}
ax.barh(dtypes_memory.keys(), dtypes_memory.values(), color="steelblue", edgecolor="black", linewidth=1)
ax.set_xlabel("Memory (MB)"); ax.set_title("Estimated Memory Usage", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 11.4: Data completeness summary
ax = axes[1, 1]
completeness = {
    "Sales": 100 - zero_pct,
    "Prices": price_cnt / nrows_total * 100,
    "Dates": 100,
    "Items": 100,
    "Stores": 100
}
colors_complete = ["red" if v < 80 else "yellow" if v < 95 else "green" for v in completeness.values()]
ax.bar(completeness.keys(), completeness.values(), color=colors_complete, edgecolor="black", linewidth=1)
ax.set_ylabel("Completeness %"); ax.set_title("Data Completeness", fontweight="bold")
ax.set_ylim([0, 105])
ax.axhline(95, color="orange", ls="--", lw=1, label="95% threshold")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("09_data_quality.png")
plt.close()
print("  ✓ Saved: 09_data_quality.png")

# ============================================================================
# [12] COMPREHENSIVE SUMMARY REPORT
# ============================================================================
print("\n[12] GENERATING SUMMARY REPORT...")

summary_data = {
    "Category": [
        "VOLUME","VOLUME","VOLUME","VOLUME","VOLUME","VOLUME","VOLUME",
        "DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION",
        "SEASONALITY","SEASONALITY","SEASONALITY","SEASONALITY",
        "EVENTS","EVENTS","EVENTS",
        "PRICING","PRICING","PRICING","PRICING",
        "SPARSITY","SPARSITY","SPARSITY","SPARSITY",
        "QUALITY","QUALITY","QUALITY",
        "HIERARCHY","HIERARCHY","HIERARCHY","HIERARCHY"
    ],
    "Metric": [
        "Total Rows","Total Items","Total Stores","Total Categories","Total Departments","Date Range (days)","Total Sales (units)",
        "Mean Sales","Std Dev Sales","Min Sales","Max Sales","Skewness","Coeff of Variation",
        "Best Day of Week","Worst Day of Week","Best Month","Worst Month",
        "Event Lift (%)","Events (Type 1) Count","SNAP Impact (avg)",
        "Mean Price","Price Std Dev","Price-Sales Corr","Price Volatility (top item)",
        "Zero Sales %","Avg Item Zero %","Items with >75% zeros","Avg Item Velocity",
        "Date Coverage %","Price Coverage %","Anomalies (z>2.5)",
        "Stores per Item (avg)","Items per Store (avg)","State with Most Sales","Top Category by Sales"
    ],
    "Value": [
        f"{nrows_total:,}",
        f"{n_items:,}",
        f"{n_stores}",
        f"{n_cats}",
        f"{n_depts}",
        f"{(train_sample['date'].max() - train_sample['date'].min()).days}",
        f"{sales_sum:,.0f}",
        f"{sales_mean:.4f}",
        f"{sales_std:.4f}",
        f"{sales_min:.4f}",
        f"{sales_max:.4f}",
        f"{sales_skew_est:.4f}",
        f"{sales_std/sales_mean:.4f}" if sales_mean > 0 else "N/A",
        dow_labels[np.argmax(dow_vals)],
        dow_labels[np.argmin(dow_vals)],
        labels_m[np.argmax(month_vals)],
        labels_m[np.argmin(month_vals)],
        f"{event_lift:+.2f}%",
        f"{sum(event_type_1_cnt.values()):,}",
        f"{(event_avg / base_avg - 1) * 100:+.2f}%" if base_avg > 0 else "N/A",
        f"{price_mean:.2f}",
        f"{price_std:.2f}",
        f"{corr_coef:.4f}",
        f"{max(item_price_vol.values()) if item_price_vol else 0:.2f}",
        f"{zero_pct:.2f}%",
        f"{items_zero_pct.mean():.2f}%",
        f"{(item_zero_pcts > 75).sum():,}",
        f"{items_velocity.mean():.4f}",
        f"{coverage:.2f}%" if expected_rows > 0 else "N/A",
        f"{price_cnt / nrows_total * 100:.2f}%",
        f"{daily_copy['anomaly'].sum()}",
        f"{stores_per_item.mean():.1f}",
        f"{items_per_store.mean():.0f}",
        sorted(state_sales.items(), key=lambda x: x[1], reverse=True)[0][0],
        sorted(cat_sales.items(), key=lambda x: x[1], reverse=True)[0][0],
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "eda_summary.csv"), index=False)
print(f"  ✓ Saved: eda_summary.csv")

print("\n" + "=" * 90)
print("SUMMARY STATISTICS")
print("=" * 90)
for category in summary_df["Category"].unique():
    print(f"\n  {category}:")
    cat_df = summary_df[summary_df["Category"] == category]
    for _, row in cat_df.iterrows():
        print(f"    {row['Metric']:.<40} {row['Value']:>20}")

# ============================================================================
# [13] ADVANCED QUALITY CHECKS
# ============================================================================
print("\n[13] ADVANCED QUALITY CHECKS...")

dup_count = train_sample.duplicated().sum()
dup_key_count = train_sample.duplicated(subset=["id", "date", "store_id"]).sum()
print(f"  ✓ Duplicates (full row, sample): {dup_count}")
print(f"  ✓ Duplicates (id-date-store, sample): {dup_key_count}")

# ============================================================================
# [14] FEATURE RELATIONSHIPS
# ============================================================================
print("\n[14] FEATURE RELATIONSHIPS...")

num_cols = ["sales", "sell_price", "wday", "month", "year", "snap_CA", "snap_TX", "snap_WI"]
corr_df = train_sample[num_cols].dropna()
corr = corr_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap (numeric features)", fontweight="bold")
SAVE("10_corr_heatmap.png")
plt.close()
print("  ✓ Saved: 10_corr_heatmap.png")

pivot_sc = train_sample.pivot_table(index="store_id", columns="cat_id", values="sales", aggfunc="mean")
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_sc, cmap="YlGnBu")
plt.title("Avg Sales Heatmap: Store x Category", fontweight="bold")
SAVE("11_store_cat_heatmap.png")
plt.close()
print("  ✓ Saved: 11_store_cat_heatmap.png")

# ============================================================================
# [15] STATIONARITY & DECOMPOSITION
# ============================================================================
print("\n[15] STATIONARITY & DECOMPOSITION...")

series = daily_sorted["total_sales"].dropna()

adf_res = adfuller(series, autolag="AIC")
kpss_res = kpss(series, regression="c", nlags="auto")

print(f"  ✓ ADF p-value:  {adf_res[1]:.6f}")
print(f"  ✓ KPSS p-value: {kpss_res[1]:.6f}")

try:
    decomp = seasonal_decompose(series, model="additive", period=7)
    fig = decomp.plot()
    fig.set_size_inches(14, 8)
    plt.tight_layout()
    SAVE("12_seasonal_decompose.png")
    plt.close()
    print("  ✓ Saved: 12_seasonal_decompose.png")
except Exception as e:
    print(f"  ! Decompose failed: {e}")

# ============================================================================
# [16] LAG ANALYSIS
# ============================================================================
print("\n[16] LAG ANALYSIS...")

lags = range(1, 29)
lag_corrs = [series.autocorr(lag=i) for i in lags]

plt.figure(figsize=(10, 4))
plt.bar(lags, lag_corrs, color="steelblue", edgecolor="black")
plt.axhline(0, color="black", lw=0.8)
plt.title("Lag Correlations (1-28)", fontweight="bold")
plt.xlabel("Lag (days)")
plt.ylabel("Correlation")
SAVE("13_lag_corr.png")
plt.close()
print("  ✓ Saved: 13_lag_corr.png")

# ============================================================================
# [17] SPARSE ITEM TIME SERIES
# ============================================================================
print("\n[17] SPARSE ITEM TIME SERIES...")

top_sparse_items = sorted(item_zero.items(), key=lambda x: x[1], reverse=True)[:5]
plt.figure(figsize=(14, 6))
for item_id, _ in top_sparse_items:
    item_ts = train_sample[train_sample["id"] == item_id].groupby("date")["sales"].sum()
    plt.plot(item_ts.index, item_ts.values, lw=1, label=item_id, alpha=0.8)
plt.title("Top Sparse Items - Sales Over Time", fontweight="bold")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
SAVE("14_sparse_items_ts.png")
plt.close()
print("  ✓ Saved: 14_sparse_items_ts.png")

print("\n" + "=" * 90)
print("✓ EDA COMPLETE! (FULL)")
print("=" * 90)
print(f"  Figures saved to: {FIGURES_DIR}")
print(f"  Summary saved to: {OUTPUT_DIR}")
print("=" * 90 + "\n")

In [ ]:
"""
M5 Forecasting — Kaggle 30GB SAFE Pipeline
============================================
Global model (1 model cho tất cả 10 stores), KHÔNG tràn RAM.

Chiến lược chống tràn:
  ▸ STEP A: Đọc CSV theo CHUNK 500k rows, encode categorical ngay
            khi đọc, ghi thẳng ra parquet (nén Snappy).
            Peak RAM của 1 chunk: ~500 MB.

  ▸ STEP B: Đọc parquet lại (lazy, dtype nhỏ).
            Toàn bộ data ~3-4 GB trong RAM.

  ▸ STEP C: Feature engineering PER-STORE (10 stores).
            Mỗi store ~6M rows, peak ~3 GB.
            Sau khi tính xong features cho store đó →
            ghi thẳng vào memmap → del store DataFrame.

  ▸ STEP D: Train từ memmap read-only.
            Không load lại vào RAM.

Peak RAM ước tính:
  - Load chunk:     <2 GB
  - FE per-store:   ~5 GB
  - Train LGB:      ~12 GB
  - Train CB:       ~14 GB
  - Train XGB:      ~12 GB
"""

import os, gc, warnings, time, json
warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import psutil
from pathlib import Path

import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("/kaggle/input/datasets/minhduylebao/dataeda")
WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "output"
FIGURES_DIR = WORK_DIR / "figures"
MODELS_DIR  = WORK_DIR / "models"
MMAP_DIR    = WORK_DIR / "mmap"
PARQUET_DIR = WORK_DIR / "parquet"
for d in [OUTPUT_DIR, FIGURES_DIR, MODELS_DIR, MMAP_DIR, PARQUET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HORIZON     = 28
LAGS        = [1, 7, 14, 28]
USE_LAG_364 = False        # Tắt cho an toàn RAM. Bật khi RAM dư.
USE_EWM     = True
CHUNK_SIZE  = 500_000      # Số rows / chunk khi đọc CSV
N_BOOST     = 1500
EARLY_STOP  = 100

# ─────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────
def ram_now():
    m = psutil.virtual_memory()
    return (f"RAM {m.used/1e9:>5.1f}/{m.total/1e9:.1f}GB "
            f"({m.percent:>4.1f}%) free={m.available/1e9:.1f}GB")

def log(msg, indent=0):
    print(f"{'  '*indent}{msg}")

def banner(s):
    print("\n" + "═"*70)
    print(f"  {s}")
    print("═"*70)
    print(f"  {ram_now()}")

def force_gc():
    for _ in range(3):
        gc.collect()

t0 = time.time()
banner("M5 FORECASTING — KAGGLE 30GB SAFE PIPELINE")


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP A. CHUNKED LOAD → PARQUET                                   ║
# ║  Đọc CSV 500k rows/lần, encode ngay, ghi parquet                  ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[A] CHUNKED LOAD → PARQUET")

# Schema mong đợi
WANT_COLS = [
    "id","item_id","dept_id","cat_id","store_id","state_id",
    "d","day_id","date",
    "sales","sell_price",
    "wday","month","year",
    "snap_CA","snap_TX","snap_WI",
    "event_type_1","event_type_2",
]

# Dtypes khi đọc — string cho categorical (factorize sau)
READ_DTYPES = {
    "day_id":     "Int32",
    "sales":      "float32",
    "sell_price": "float32",
    "wday":       "Int8",
    "month":      "Int8",
    "year":       "Int16",
    "snap_CA":    "Int8",
    "snap_TX":    "Int8",
    "snap_WI":    "Int8",
}

EVENT_MAP = {"None":0, "Cultural":1, "National":2, "Religious":3, "Sporting":4}


def discover_csv(fname):
    """Tìm file CSV cho mỗi split."""
    p = DATA_DIR / fname
    if p.exists():
        return p
    alts = list(DATA_DIR.glob(f"{fname.split('.')[0]}*.csv"))
    if alts:
        return alts[0]
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR}")


def csv_columns(path):
    """Lấy danh sách columns mà không load data."""
    return pd.read_csv(path, nrows=0).columns.tolist()


# ────────────────────────────────────────────────────────────────────
# PHASE A1: Quét toàn bộ data để xây global mapping cho id, item_id,
#           dept_id, cat_id, store_id, state_id
# ────────────────────────────────────────────────────────────────────
log("Phase A1: Build global categorical mappings...")

CAT_COLS = ["id","item_id","dept_id","cat_id","store_id","state_id"]
cat_uniques = {c: set() for c in CAT_COLS}

splits_info = [
    ("train.csv", 0),
    ("val.csv",   1),
    ("test.csv",  2),
]

for fname, _ in splits_info:
    path  = discover_csv(fname)
    avail = csv_columns(path)
    use   = [c for c in CAT_COLS if c in avail]
    log(f"  Scanning {fname}...", 1)
    for chunk in pd.read_csv(path, usecols=use, dtype={c:"string" for c in use},
                             chunksize=CHUNK_SIZE, engine="c", low_memory=True):
        for c in use:
            cat_uniques[c].update(chunk[c].dropna().unique().tolist())
        del chunk
    force_gc()

# Build mappings (sorted để deterministic)
cat_maps = {}
cat_dtypes = {"id":"int32","item_id":"int16","dept_id":"int8",
              "cat_id":"int8","store_id":"int8","state_id":"int8"}
for c in CAT_COLS:
    sorted_vals = sorted(cat_uniques[c])
    cat_maps[c] = {v: i for i, v in enumerate(sorted_vals)}
    log(f"  {c}: {len(sorted_vals):>6} unique → {cat_dtypes[c]}", 1)

# Lưu id mapping ngược cho submission
id_lookup = np.array(sorted(cat_uniques["id"]))  # code → string
store_id_lookup = np.array(sorted(cat_uniques["store_id"]))
N_STORES = len(store_id_lookup)
log(f"  Total stores: {N_STORES}", 1)

del cat_uniques
force_gc()
log(f"  {ram_now()}", 1)

# ────────────────────────────────────────────────────────────────────
# PHASE A2: Đọc CSV theo chunk, encode, ghi parquet
# ────────────────────────────────────────────────────────────────────
log("\nPhase A2: Chunked encode → parquet...")

def encode_chunk(df, split_tag):
    """Encode 1 chunk: cast dtype, factorize categorical, parse date."""
    # Factorize categoricals dùng global mapping
    for c in CAT_COLS:
        if c in df.columns:
            df[f"{c}_enc"] = (df[c].map(cat_maps[c])
                                   .astype(cat_dtypes[c]))
            df.drop(columns=[c], inplace=True)

    # Drop "d" nếu có (dùng day_id)
    if "d" in df.columns:
        if "day_id" not in df.columns or df["day_id"].isna().any():
            df["day_id"] = (df["d"].astype("string")
                                   .str.replace("d_","",regex=False)
                                   .astype("int32"))
        df.drop(columns=["d"], inplace=True)

    # Parse date → wday, month, year, ...
    if "date" in df.columns:
        dt = pd.to_datetime(df["date"], errors="coerce")
        if "wday"  not in df.columns: df["wday"]  = dt.dt.dayofweek.astype("Int8")
        if "month" not in df.columns: df["month"] = dt.dt.month.astype("Int8")
        if "year"  not in df.columns: df["year"]  = dt.dt.year.astype("Int16")
        df["quarter"]      = dt.dt.quarter.astype("int8")
        df["is_weekend"]   = (dt.dt.dayofweek >= 5).astype("int8")
        df["day_of_month"] = dt.dt.day.astype("int8")
        df.drop(columns=["date"], inplace=True)
        del dt

    # Encode events
    for raw in ["event_type_1","event_type_2"]:
        if raw in df.columns:
            df[raw+"_enc"] = (df[raw].fillna("None").map(EVENT_MAP)
                                    .fillna(0).astype("int8"))
            df.drop(columns=[raw], inplace=True)
        else:
            df[raw+"_enc"] = np.int8(0)

    # Drop event_name nếu có
    for c in ["event_name_1","event_name_2","wm_yr_wk"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

    # snap_*: fill 0
    for s in ["CA","TX","WI"]:
        col = f"snap_{s}"
        if col not in df.columns:
            df[col] = np.int8(0)
        else:
            df[col] = df[col].fillna(0).astype("int8")

    # sales
    if "sales" not in df.columns:
        df["sales"] = np.float32(np.nan)
    # Train: NaN sales = 0 (out-of-stock)
    if split_tag == 0:
        df["sales"] = df["sales"].fillna(0).astype("float32")
    else:
        df["sales"] = df["sales"].astype("float32")

    # sell_price NaN → keep, ffill sau
    if "sell_price" not in df.columns:
        df["sell_price"] = np.float32(np.nan)
    df["sell_price"] = df["sell_price"].astype("float32")

    # Cast các int columns về int8 cuối cùng
    for c in ["wday","month","snap_CA","snap_TX","snap_WI"]:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int8")
    if "year" in df.columns:
        df["year"] = (df["year"].fillna(2011).astype("int32") - 2010).astype("int8")

    df["_split"] = np.int8(split_tag)
    return df


def write_split_parquet(fname, split_tag, parquet_path):
    """Đọc CSV theo chunk, encode, ghi parquet."""
    path  = discover_csv(fname)
    avail = csv_columns(path)
    use   = [c for c in WANT_COLS if c in avail]
    dt    = {k:v for k,v in READ_DTYPES.items() if k in use}
    # Đọc categoricals dưới dạng string để map về int qua cat_maps
    for c in CAT_COLS:
        if c in use:
            dt[c] = "string"

    # PyArrow writer để append từng chunk
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
        writer = None
        n_total = 0
        for chunk_idx, chunk in enumerate(pd.read_csv(
                path, usecols=use, dtype=dt,
                chunksize=CHUNK_SIZE, engine="c", low_memory=True)):
            chunk = encode_chunk(chunk, split_tag)
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema,
                                           compression="snappy")
            writer.write_table(table)
            n_total += len(chunk)
            del chunk, table
            force_gc()
            log(f"  chunk {chunk_idx+1}: rows={n_total:>10,} | {ram_now()}", 2)
        if writer is not None:
            writer.close()
    except ImportError:
        # Fallback: load all then write (kém an toàn hơn)
        log("  (no pyarrow, using pandas write)", 2)
        parts = []
        for chunk in pd.read_csv(path, usecols=use, dtype=dt,
                                  chunksize=CHUNK_SIZE, engine="c"):
            parts.append(encode_chunk(chunk, split_tag))
        df = pd.concat(parts, ignore_index=True)
        del parts; force_gc()
        df.to_parquet(parquet_path, compression="snappy", index=False)
        del df; force_gc()


for fname, tag in splits_info:
    out = PARQUET_DIR / f"{fname.replace('.csv','')}.parquet"
    log(f"\n  → {out.name}", 1)
    write_split_parquet(fname, tag, out)
    force_gc()

log(f"\n  Parquet files written. {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP B. LOAD PARQUET LẠI (đã nén, dtype nhỏ)                     ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[B] LOAD PARQUET (compact)")

parts = []
for fname, _ in splits_info:
    p = PARQUET_DIR / f"{fname.replace('.csv','')}.parquet"
    df_ = pd.read_parquet(p)
    parts.append(df_)
    log(f"  {p.name}: {df_.shape}  | {ram_now()}", 1)
    del df_

full = pd.concat(parts, ignore_index=True, copy=False)
del parts
force_gc()
log(f"  Concatenated: {full.shape}  | {ram_now()}", 1)

# Sort 1 lần, dùng kind='stable'
full.sort_values(["id_enc","day_id"], inplace=True, kind="stable")
full.reset_index(drop=True, inplace=True)
force_gc()
log(f"  Sorted. {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP C. FEATURE ENGINEERING (per-store chunks)                   ║
# ║  Tính features cho từng store riêng → ghi memmap → xóa            ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[C] FEATURE ENGINEERING — PER STORE")

# ───── Trước hết: các features không phụ thuộc lag/rolling (rẻ) ─────
# Làm trên toàn bộ full để dùng cho price stats

log("Phase C0: Global preprocess (cheap features)...")

# sell_price ffill/bfill per (item, store) — KHÔNG dùng groupby toàn bộ
# Thay vì transform, dùng sort + fill theo group
full.sort_values(["id_enc","day_id"], inplace=True, kind="stable")
# ffill within id
full["sell_price"] = full.groupby("id_enc", sort=False, observed=True)["sell_price"]\
                         .transform(lambda x: x.ffill().bfill())
full["sell_price"] = full["sell_price"].fillna(0).astype("float32")
force_gc()
log(f"  sell_price filled  | {ram_now()}", 1)

# Event interaction (cheap)
full["has_event"] = ((full["event_type_1_enc"]>0) |
                     (full["event_type_2_enc"]>0)).astype("int8")
for s in ["CA","TX","WI"]:
    full[f"snap_{s}_x_event"] = (full[f"snap_{s}"] * full["has_event"]).astype("int8")
    full[f"wday_x_snap_{s}"]  = (full["wday"]      * full[f"snap_{s}"]).astype("int8")
if "day_of_month" in full.columns:
    full["is_payday"] = full["day_of_month"].isin([1,15,28,29,30,31]).astype("int8")
force_gc()

# Price stats từ train+val
log("Phase C0: Price stats...", 1)
mask_known = full["_split"] < 2
ps = (full.loc[mask_known]
          .groupby(["item_id_enc","store_id_enc"], observed=True)["sell_price"]
          .agg(["mean","min","max"]))
del mask_known
force_gc()

mi = pd.MultiIndex.from_arrays([full["item_id_enc"], full["store_id_enc"]])
pmean = mi.map(ps["mean"]).astype("float32")
full["price_ratio"] = (full["sell_price"]/(pmean+1e-9)).astype("float32")
del pmean; force_gc()

pmin = mi.map(ps["min"]).astype("float32")
pmax = mi.map(ps["max"]).astype("float32")
full["price_norm"] = ((full["sell_price"]-pmin)/(pmax-pmin+1e-9)).astype("float32")
del pmin, pmax, ps, mi; force_gc()
log(f"  Price stats done  | {ram_now()}", 1)

# Item global mean
log("Phase C0: Item global mean...", 1)
im = full.loc[full["_split"]<2].groupby("item_id_enc", observed=True)["sales"].mean()
full["item_gmean"] = full["item_id_enc"].map(im).fillna(0).astype("float32")
del im; force_gc()
log(f"  Item gmean done  | {ram_now()}", 1)

# Aggregate features: store_day_mean, dept_day_mean, cat_day_mean, global_day_mean
# Tính trên train+val rồi map về full để tránh leak
log("Phase C0: Aggregate day means...", 1)
for keys, name in [
    (["store_id_enc","day_id"], "store_day_mean"),
    (["dept_id_enc","day_id"],  "dept_day_mean"),
    (["cat_id_enc","day_id"],   "cat_day_mean"),
    (["day_id"],                "global_day_mean"),
]:
    agg = (full.loc[full["_split"]<2]
                .groupby(keys, observed=True)["sales"].mean())
    if len(keys) == 1:
        full[name] = (full[keys[0]].map(agg)
                                    .fillna(0).astype("float32"))
    else:
        mi2 = pd.MultiIndex.from_arrays([full[k] for k in keys])
        full[name] = mi2.map(agg).fillna(0).astype("float32")
        del mi2
    del agg
    force_gc()
    log(f"  {name} ✓  | {ram_now()}", 2)


# ───── Phase C1: Lag/rolling per-store, write to memmap ─────
log("\nPhase C1: Lag/rolling per-store + memmap write...")

LAG_COLS = [f"lag_{l}" for l in LAGS]
if USE_LAG_364:
    LAG_COLS.append("lag_364")

ROLL_COLS = ["rmean_7","rmean_14","rmean_28","rstd_28","rmax_7","lag1_diff","trend_7"]
if USE_EWM:
    ROLL_COLS += ["ewm_01","ewm_05"]

# Sẽ build danh sách columns features sau cùng
# Trước hết define các cột KHÔNG phải feature
NOT_FEATURE = {"sales", "_split", "day_id"}  # giữ day_id để filter, drop khi train

# Build danh sách cột "cheap" (đã có sẵn trong full)
CHEAP_FEATS = [c for c in full.columns if c not in NOT_FEATURE]
log(f"  Cheap features: {len(CHEAP_FEATS)}", 1)

# Vì lag/rolling cần shift theo từng id, ta phải tính per-id
# Tối ưu: tính per store_id (group lớn hơn id) trên 1 subset nhỏ của full

# Trước hết: tạo masks toàn cục
min_day = max(LAGS) + 1
log(f"  min_day for train: {min_day}", 1)

# Pre-compute MASK theo split + day_id
tr_mask_all  = ((full["_split"]==0) & (full["day_id"]>=min_day)).to_numpy()
val_mask_all = (full["_split"]==1).to_numpy()
te_mask_all  = (full["_split"]==2).to_numpy()

N_TR  = int(tr_mask_all.sum())
N_VAL = int(val_mask_all.sum())
N_TE  = int(te_mask_all.sum())
log(f"  Rows: tr={N_TR:,} val={N_VAL:,} te={N_TE:,}", 1)

# Save metadata để build submission sau
test_id_codes = full.loc[te_mask_all, "id_enc"].to_numpy().astype(np.int32)
test_days     = full.loc[te_mask_all, "day_id"].to_numpy().astype(np.int32)
np.save(MMAP_DIR / "test_id_codes.npy", test_id_codes)
np.save(MMAP_DIR / "test_days.npy",     test_days)
np.save(MMAP_DIR / "id_lookup.npy",     id_lookup)

# Save y
y_tr_arr  = full.loc[tr_mask_all,  "sales"].to_numpy().astype(np.float32)
y_val_arr = full.loc[val_mask_all, "sales"].to_numpy().astype(np.float32)

# ───── Tính lag/rolling per-id trên TOÀN BỘ full (1 groupby duy nhất) ─────
log("  Computing lag features (1 groupby)...", 1)
g_sales = full.groupby("id_enc", sort=False, observed=True)["sales"]

# Add zero_streak FIRST (cần thiết)
iz = (full["sales"].fillna(0) == 0).astype("int8")
grp_id = full["id_enc"].to_numpy()
# zero_streak: chuỗi 0 liên tiếp
streak = np.zeros(len(full), dtype=np.int16)
prev_id = -1
cnt = 0
iz_arr = iz.to_numpy()
for i in range(len(full)):
    if grp_id[i] != prev_id:
        cnt = 0
        prev_id = grp_id[i]
    if iz_arr[i] == 1:
        cnt += 1
    else:
        cnt = 0
    streak[i] = cnt
full["zero_streak"] = streak
del iz, iz_arr, streak, grp_id
force_gc()
log(f"    zero_streak ✓  | {ram_now()}", 2)

# Lags
for lag in LAGS:
    full[f"lag_{lag}"] = g_sales.shift(lag).astype("float32")
    force_gc()
    log(f"    lag_{lag} ✓  | {ram_now()}", 2)

if USE_LAG_364:
    full["lag_364"] = g_sales.shift(364).astype("float32")
    force_gc()
    log(f"    lag_364 ✓  | {ram_now()}", 2)

# Rolling cần shift(1) reuse
log("  Computing rolling features...", 1)
s1 = g_sales.shift(1)  # shift-1, dùng làm base

for w in [7, 14, 28]:
    full[f"rmean_{w}"] = s1.rolling(w, min_periods=1).mean().astype("float32")
    force_gc()
    log(f"    rmean_{w} ✓  | {ram_now()}", 2)

full["rstd_28"] = s1.rolling(28, min_periods=1).std().fillna(0).astype("float32")
force_gc()
full["rmax_7"]  = s1.rolling(7, min_periods=1).max().fillna(0).astype("float32")
force_gc()

if USE_EWM:
    full["ewm_01"] = s1.ewm(alpha=0.1, min_periods=1).mean().astype("float32")
    full["ewm_05"] = s1.ewm(alpha=0.5, min_periods=1).mean().astype("float32")
    force_gc()
    log(f"    ewm done  | {ram_now()}", 2)

full["lag1_diff"] = s1.diff().fillna(0).astype("float32")
del s1, g_sales
force_gc()

full["trend_7"]  = (full["lag_1"]-full["lag_7"]).fillna(0).astype("float32")

# Relative features
full["vs_store"] = (full["lag_1"]/(full["store_day_mean"]+1e-9)).clip(-10,10).astype("float32")
full["vs_item"]  = (full["lag_1"]/(full["item_gmean"]+1e-9)).clip(-10,10).astype("float32")
force_gc()

log(f"  All features done. Shape: {full.shape} | {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP D. WRITE TO MEMMAP (stream column-by-column)                ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[D] WRITE MEMMAPS (stream by column)")

# Build final feature list
FCOLS = [c for c in full.columns if c not in NOT_FEATURE]
N_FEAT = len(FCOLS)
log(f"Total features: {N_FEAT}", 1)

# Lưu feature list
with open(MMAP_DIR / "feature_cols.json", "w") as f:
    json.dump(FCOLS, f)

# Open memmaps in write mode
X_tr_path  = MMAP_DIR / "X_tr.dat"
X_val_path = MMAP_DIR / "X_val.dat"
X_te_path  = MMAP_DIR / "X_te.dat"
y_tr_path  = MMAP_DIR / "y_tr.dat"
y_val_path = MMAP_DIR / "y_val.dat"

# Calculate size required
size_tr_gb  = N_TR  * N_FEAT * 4 / 1e9
size_val_gb = N_VAL * N_FEAT * 4 / 1e9
size_te_gb  = N_TE  * N_FEAT * 4 / 1e9
log(f"Memmap sizes: tr={size_tr_gb:.2f}GB val={size_val_gb:.2f}GB te={size_te_gb:.2f}GB", 1)
log(f"Total disk: {size_tr_gb+size_val_gb+size_te_gb:.2f}GB", 1)

X_tr_mm  = np.memmap(X_tr_path,  dtype="float32", mode="w+", shape=(N_TR,  N_FEAT))
X_val_mm = np.memmap(X_val_path, dtype="float32", mode="w+", shape=(N_VAL, N_FEAT))
X_te_mm  = np.memmap(X_te_path,  dtype="float32", mode="w+", shape=(N_TE,  N_FEAT))

# Stream từng cột → write 3 slices → drop column khỏi full
log("Streaming columns to memmap...", 1)
for i, col in enumerate(FCOLS):
    v = full[col].to_numpy(dtype=np.float32, copy=False)
    X_tr_mm[:,  i] = v[tr_mask_all]
    X_val_mm[:, i] = v[val_mask_all]
    X_te_mm[:,  i] = v[te_mask_all]
    del v
    # Drop column khỏi full để giải phóng RAM
    full.drop(columns=[col], inplace=True)
    if (i+1) % 10 == 0 or (i+1) == N_FEAT:
        X_tr_mm.flush(); X_val_mm.flush(); X_te_mm.flush()
        force_gc()
        log(f"  [{i+1:>3}/{N_FEAT}] last: {col} | {ram_now()}", 2)

# Write y
y_tr_mm  = np.memmap(y_tr_path,  dtype="float32", mode="w+", shape=(N_TR,))
y_val_mm = np.memmap(y_val_path, dtype="float32", mode="w+", shape=(N_VAL,))
y_tr_mm[:]  = y_tr_arr
y_val_mm[:] = y_val_arr
y_tr_mm.flush(); y_val_mm.flush()

# CLEANUP — quan trọng nhất, xóa hết DataFrame trước khi train
del full, X_tr_mm, X_val_mm, X_te_mm, y_tr_mm, y_val_mm
del tr_mask_all, val_mask_all, te_mask_all
del y_tr_arr, y_val_arr
del cat_maps
force_gc()
log(f"All cleanup done. {ram_now()}", 1)

# Re-open memmaps read-only
X_tr  = np.memmap(X_tr_path,  dtype="float32", mode="r", shape=(N_TR,  N_FEAT))
X_val = np.memmap(X_val_path, dtype="float32", mode="r", shape=(N_VAL, N_FEAT))
X_te  = np.memmap(X_te_path,  dtype="float32", mode="r", shape=(N_TE,  N_FEAT))
y_tr  = np.memmap(y_tr_path,  dtype="float32", mode="r", shape=(N_TR,))
y_val = np.memmap(y_val_path, dtype="float32", mode="r", shape=(N_VAL,))

log(f"X_tr: {X_tr.shape}", 1)
log(f"X_val: {X_val.shape}", 1)
log(f"X_te: {X_te.shape}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP E. TRAIN — GLOBAL MODEL (1 model cho all stores)            ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[E1] LIGHTGBM (GLOBAL)")

# LightGBM cần data trong RAM cho speed. Convert memmap → numpy 1 lần.
# Đây là điểm RAM cao nhất.
log("Converting memmap → numpy for LGB...")
X_tr_np  = np.array(X_tr,  dtype=np.float32)
X_val_np = np.array(X_val, dtype=np.float32)
y_tr_np  = np.array(y_tr,  dtype=np.float32)
y_val_np = np.array(y_val, dtype=np.float32)
log(f"Loaded into RAM. {ram_now()}", 1)

lgb_params = dict(
    objective="tweedie", tweedie_variance_power=1.1,
    metric="rmse", boosting_type="gbdt",
    learning_rate=0.05, num_leaves=127,
    feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1,
    min_child_samples=50, reg_alpha=0.1, reg_lambda=1.0,
    seed=42, verbose=-1, max_bin=255,
)

# GPU first, fallback CPU
def train_lgb(params):
    dtr = lgb.Dataset(X_tr_np, y_tr_np, feature_name=FCOLS,
                      free_raw_data=False)
    dvl = lgb.Dataset(X_val_np, y_val_np, reference=dtr,
                      free_raw_data=False)
    m = lgb.train(params, dtr, num_boost_round=N_BOOST,
                  valid_sets=[dvl],
                  callbacks=[lgb.early_stopping(EARLY_STOP),
                             lgb.log_evaluation(200)])
    del dtr, dvl
    return m

try:
    params_gpu = {**lgb_params, "device":"gpu",
                  "gpu_platform_id":0, "gpu_device_id":0}
    lgb_m = train_lgb(params_gpu)
    log("LGB trained on GPU", 1)
except Exception as e:
    log(f"GPU failed ({e}); fallback CPU", 1)
    lgb_m = train_lgb(lgb_params)

lgb_vp = lgb_m.predict(X_val_np).astype(np.float32)

# Predict test: memmap-based để tránh load
X_te_np = np.array(X_te, dtype=np.float32)
lgb_tp = lgb_m.predict(X_te_np).astype(np.float32)
del X_te_np
force_gc()

lgb_rmse = float(np.sqrt(mean_squared_error(y_val_np, lgb_vp)))
log(f"LGB RMSE: {lgb_rmse:.4f}", 1)

lgb_m.save_model(str(MODELS_DIR/"lgb.txt"))
fi = pd.DataFrame({"feature":FCOLS,
                    "gain":lgb_m.feature_importance("gain")})\
        .sort_values("gain", ascending=False)
fi.to_csv(OUTPUT_DIR/"lgb_importance.csv", index=False)
log(f"Top 10:\n{fi.head(10)[['feature','gain']].to_string(index=False)}", 1)

del lgb_m; force_gc()
log(f"After LGB. {ram_now()}", 1)


# ─── CatBoost ───
banner("[E2] CATBOOST (GLOBAL)")

def train_cb(task_type):
    m = cb.CatBoostRegressor(
        loss_function="RMSE", eval_metric="RMSE",
        iterations=N_BOOST, learning_rate=0.05, depth=8,
        l2_leaf_reg=3, min_data_in_leaf=50,
        random_seed=42, verbose=200, early_stopping_rounds=EARLY_STOP,
        **({"task_type":"GPU", "devices":"0",
            "subsample":0.7, "colsample_bylevel":0.7}
           if task_type=="GPU"
           else {"thread_count":-1}),
    )
    m.fit(X_tr_np, y_tr_np,
          eval_set=(X_val_np, y_val_np), feature_names=FCOLS)
    return m

try:
    cb_m = train_cb("GPU")
    log("CB trained on GPU", 1)
except Exception as e:
    log(f"GPU failed ({e}); fallback CPU", 1)
    cb_m = train_cb("CPU")

cb_vp = cb_m.predict(X_val_np).astype(np.float32)
X_te_np = np.array(X_te, dtype=np.float32)
cb_tp = cb_m.predict(X_te_np).astype(np.float32)
del X_te_np; force_gc()

cb_rmse = float(np.sqrt(mean_squared_error(y_val_np, cb_vp)))
log(f"CB RMSE: {cb_rmse:.4f}", 1)
cb_m.save_model(str(MODELS_DIR/"catboost.cbm"))
del cb_m; force_gc()
log(f"After CB. {ram_now()}", 1)


# ─── XGBoost ───
banner("[E3] XGBOOST (GLOBAL)")

def train_xgb(device):
    m = xgb.XGBRegressor(
        n_estimators=N_BOOST, learning_rate=0.05,
        max_depth=8, subsample=0.7, colsample_bytree=0.7,
        min_child_weight=50, reg_alpha=0.1, reg_lambda=1.0,
        objective="reg:tweedie", tweedie_variance_power=1.1,
        tree_method="hist", device=device,
        random_state=42, early_stopping_rounds=EARLY_STOP,
        eval_metric="rmse", verbosity=1,
        n_jobs=-1 if device=="cpu" else 1,
    )
    m.fit(X_tr_np, y_tr_np,
          eval_set=[(X_val_np, y_val_np)], verbose=200)
    return m

try:
    xgb_m = train_xgb("cuda")
    log("XGB trained on GPU", 1)
except Exception as e:
    log(f"GPU failed ({e}); fallback CPU", 1)
    xgb_m = train_xgb("cpu")

xgb_vp = xgb_m.predict(X_val_np).astype(np.float32)
X_te_np = np.array(X_te, dtype=np.float32)
xgb_tp = xgb_m.predict(X_te_np).astype(np.float32)
del X_te_np; force_gc()

xgb_rmse = float(np.sqrt(mean_squared_error(y_val_np, xgb_vp)))
log(f"XGB RMSE: {xgb_rmse:.4f}", 1)
xgb_m.save_model(str(MODELS_DIR/"xgb.json"))
del xgb_m, X_tr_np, X_val_np; force_gc()
log(f"After XGB. {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP F. ENSEMBLE + METRICS + SUBMISSION                          ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[F1] ENSEMBLE")

sc = np.array([lgb_rmse, cb_rmse, xgb_rmse], dtype=np.float64)
w  = (1.0/sc); w = w/w.sum()
log(f"Weights: LGB={w[0]:.3f}  CB={w[1]:.3f}  XGB={w[2]:.3f}", 1)

ens_vp = (w[0]*lgb_vp + w[1]*cb_vp + w[2]*xgb_vp).astype(np.float32)
ens_tp = (w[0]*lgb_tp + w[1]*cb_tp + w[2]*xgb_tp).astype(np.float32)


banner("[F2] METRICS")

def metrics(yt, yp, ynaive=None, name=""):
    yt = np.asarray(yt, dtype=np.float64)
    yp = np.asarray(yp, dtype=np.float64)
    mae  = float(np.mean(np.abs(yt-yp)))
    rmse = float(np.sqrt(np.mean((yt-yp)**2)))
    wape = float(np.sum(np.abs(yt-yp))/(np.sum(np.abs(yt))+1e-9)*100)
    bias = float(np.mean(yp-yt)/(np.mean(yt)+1e-9)*100)
    r2   = float(r2_score(yt, yp))
    if ynaive is not None:
        mase = float(mae/(np.mean(np.abs(np.diff(ynaive.astype(np.float64))))+1e-9))
    else:
        mase = np.nan
    return {"Model":name, "MAE":round(mae,4), "RMSE":round(rmse,4),
            "WAPE%":round(wape,4), "Bias%":round(bias,4),
            "MASE":round(mase,4) if not np.isnan(mase) else "N/A",
            "R2":round(r2,4)}

mdf = pd.DataFrame([
    metrics(y_val_np, lgb_vp, y_tr_np, "LightGBM"),
    metrics(y_val_np, cb_vp,  y_tr_np, "CatBoost"),
    metrics(y_val_np, xgb_vp, y_tr_np, "XGBoost"),
    metrics(y_val_np, ens_vp, y_tr_np, "Ensemble"),
]).set_index("Model")

print()
print(mdf.to_string())
mdf.to_csv(OUTPUT_DIR/"metrics.csv")

# Plot
COLORS = ["#4C72B0","#DD8452","#55A868","#C44E52"]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("M5 Validation — Global Model", fontweight="bold")
for a, col, ttl in zip(ax, ["RMSE","MAE","R2"], ["RMSE ↓","MAE ↓","R² ↑"]):
    v = mdf[col].astype(float).values
    bars = a.bar(mdf.index, v, color=COLORS)
    a.set_title(ttl)
    for b, val in zip(bars, v):
        a.text(b.get_x()+b.get_width()/2, val*1.005,
               f"{val:.3f}", ha="center", fontsize=8)
    a.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR/"results.png", dpi=120, bbox_inches="tight")
plt.close(fig)
log(f"Plot saved", 1)

del y_tr_np, y_val_np; force_gc()


# ─── Submission ───
banner("[F3] SUBMISSION")

pred = np.clip(ens_tp, 0.0, None).astype(np.float32)

# Reconstruct id strings
test_id_strs = id_lookup[test_id_codes]

sdf = pd.DataFrame({
    "id":     test_id_strs,
    "day_id": test_days,
    "pred":   pred,
}).sort_values(["id","day_id"]).reset_index(drop=True)

# Pivot to wide F1..F28
sdf["fcol"] = "F" + (sdf.groupby("id").cumcount()+1).astype(str)
sub = (sdf.pivot(index="id", columns="fcol", values="pred")
          .reindex(columns=[f"F{i}" for i in range(1, HORIZON+1)])
          .fillna(0).round(4).reset_index())

sub.to_csv(OUTPUT_DIR/"submission.csv", index=False)

fcheck = [f"F{i}" for i in range(1, HORIZON+1)]
n_zero = int((sub[fcheck]==0).all(axis=1).sum())
log(f"Submission: {sub.shape}", 1)
log(f"All-zero rows: {n_zero}", 1)
log(f"Sample:\n{sub.head(3).to_string()}", 1)

# Cleanup
del X_tr, X_val, X_te, y_tr, y_val; force_gc()

dt = time.time() - t0
print()
print("═"*70)
print(f"  TOTAL TIME: {dt/60:.1f} min")
print(f"  {ram_now()}")
print("═"*70)
print("Done ✓")


══════════════════════════════════════════════════════════════════════
  M5 FORECASTING — KAGGLE 30GB SAFE PIPELINE
══════════════════════════════════════════════════════════════════════
  RAM   1.2/33.7GB ( 5.0%) free=32.0GB

══════════════════════════════════════════════════════════════════════
  [A] CHUNKED LOAD → PARQUET
══════════════════════════════════════════════════════════════════════
  RAM   1.2/33.7GB ( 5.0%) free=32.0GB
Phase A1: Build global categorical mappings...
    Scanning train.csv...
    Scanning val.csv...
    Scanning test.csv...
    id:  30490 unique → int32
    item_id:   3049 unique → int16
    dept_id:      7 unique → int8
    cat_id:      3 unique → int8
    store_id:     10 unique → int8
    state_id:      3 unique → int8
    Total stores: 10
    RAM   1.2/33.7GB ( 5.0%) free=32.0GB

Phase A2: Chunked encode → parquet...
  
  → train.parquet
      chunk 1: rows=   500,000 | RAM   1.3/33.7GB ( 5.2%) free=31.9GB
      chunk 2: rows= 1,000,000 | RAM   1.3/33.

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 100 rounds


[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



  GPU failed (Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .
); fallback CPU


In [1]:
"""
M5 — TRAIN ONLY v3 (RAM-SAFE)
==============================
Fix 3 vấn đề:
  1. SUBSAMPLE tại lúc load: 56M → ~20M rows (KHÔNG load full vào RAM)
     - Đọc memmap theo chunks, chỉ copy rows được chọn
     - Peak load: ~5GB thay vì 12GB
  2. CLEAN NaN/Inf: replace bằng 0 → LGB GPU không crash
  3. GPU works: sau khi clean data, LGB GPU + XGB CUDA chạy được

Bonus: feature_fraction giảm → bớt overfit
"""

import os, gc, warnings, time, json
warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import psutil
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────
WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "output"
FIGURES_DIR = WORK_DIR / "figures"
MODELS_DIR  = WORK_DIR / "models"
MMAP_DIR    = WORK_DIR / "mmap"
PRED_DIR    = WORK_DIR / "preds"
for d in [OUTPUT_DIR, FIGURES_DIR, MODELS_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HORIZON       = 28
N_BOOST       = 1200
EARLY_STOP    = 80
PREDICT_BATCH = 300_000           # giảm batch khi predict test

# ⭐ SUBSAMPLE
TRAIN_SUBSAMPLE_RATIO = 0.4    # 56M × 0.35 = ~20M rows
# Tăng nếu RAM dư, giảm nếu vẫn tràn

LOAD_CHUNK = 2_000_000            # load memmap theo 2M rows/chunk

# ─────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────
def ram_now():
    m = psutil.virtual_memory()
    return (f"RAM {m.used/1e9:>5.1f}/{m.total/1e9:.1f}GB "
            f"({m.percent:>4.1f}%) free={m.available/1e9:.1f}GB")

def log(msg, indent=0):
    print(f"{'  '*indent}{msg}")

def banner(s):
    print("\n" + "═"*70)
    print(f"  {s}")
    print("═"*70)
    print(f"  {ram_now()}")

def force_gc():
    for _ in range(3):
        gc.collect()

t0 = time.time()
banner("M5 — TRAIN ONLY v3 (RAM-SAFE)")


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 1. INSPECT MEMMAPS                                          ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[1] INSPECT")

X_tr_path  = MMAP_DIR / "X_tr.dat"
X_val_path = MMAP_DIR / "X_val.dat"
X_te_path  = MMAP_DIR / "X_te.dat"
y_tr_path  = MMAP_DIR / "y_tr.dat"
y_val_path = MMAP_DIR / "y_val.dat"

with open(MMAP_DIR/"feature_cols.json") as f:
    FCOLS = json.load(f)
N_FEAT = len(FCOLS)

def infer_n(path, n_feat=1):
    return path.stat().st_size // (n_feat * 4)

N_TR_FULL = infer_n(X_tr_path, N_FEAT)
N_VAL     = infer_n(X_val_path, N_FEAT)
N_TE      = infer_n(X_te_path,  N_FEAT)

log(f"  Features: {N_FEAT}", 1)
log(f"  N_TR_FULL: {N_TR_FULL:>10,}", 1)
log(f"  N_VAL:     {N_VAL:>10,}", 1)
log(f"  N_TE:      {N_TE:>10,}", 1)
log(f"  Subsample ratio: {TRAIN_SUBSAMPLE_RATIO}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 2. BUILD SUBSAMPLE INDICES (deterministic)                  ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[2] BUILD SUBSAMPLE INDICES")

sub_idx_path = MMAP_DIR / "train_sub_idx.npy"

if sub_idx_path.exists() and TRAIN_SUBSAMPLE_RATIO < 1.0:
    sub_idx = np.load(sub_idx_path)
    log(f"  Loaded indices: {len(sub_idx):,}", 1)
else:
    if TRAIN_SUBSAMPLE_RATIO >= 1.0:
        sub_idx = np.arange(N_TR_FULL, dtype=np.int64)
    else:
        np.random.seed(42)
        n_keep = int(N_TR_FULL * TRAIN_SUBSAMPLE_RATIO)
        # Random sample không thay thế
        sub_idx = np.random.choice(N_TR_FULL, size=n_keep, replace=False)
        sub_idx.sort()  # Sort để đọc memmap tuần tự (cache-friendly)
        sub_idx = sub_idx.astype(np.int64)
    np.save(sub_idx_path, sub_idx)
    log(f"  Built indices: {len(sub_idx):,}", 1)

N_TR = len(sub_idx)
log(f"  → N_TR after subsample: {N_TR:>10,}", 1)
log(f"  → Estimated X_tr size:  {N_TR * N_FEAT * 4 / 1e9:.2f} GB", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 3. HELPER: LOAD WITH SUBSAMPLE + CLEAN NaN                  ║
# ╚════════════════════════════════════════════════════════════════════╝

def load_X_subsampled(path, n_rows_full, n_feat, indices):
    """
    Load X từ memmap với subsample, theo chunks để tiết kiệm RAM.
    Clean NaN/Inf ngay khi load.
    Trả về np.array shape (len(indices), n_feat).
    """
    n_keep = len(indices)
    out = np.empty((n_keep, n_feat), dtype=np.float32)
    
    # Đọc memmap (lazy, không load vào RAM)
    mm = np.memmap(path, dtype="float32", mode="r",
                   shape=(n_rows_full, n_feat))
    
    # Process theo chunks of indices
    out_pos = 0
    for chunk_start in range(0, n_keep, LOAD_CHUNK):
        chunk_end = min(chunk_start + LOAD_CHUNK, n_keep)
        idx_chunk = indices[chunk_start:chunk_end]
        # Fancy indexing memmap → tạo array copy
        block = np.asarray(mm[idx_chunk])
        # Clean NaN/Inf
        np.nan_to_num(block, copy=False, nan=0.0,
                      posinf=0.0, neginf=0.0)
        out[out_pos:out_pos + len(idx_chunk)] = block
        out_pos += len(idx_chunk)
        del block
        if (chunk_start // LOAD_CHUNK) % 3 == 0:
            force_gc()
            log(f"    chunk {chunk_start:>10,}/{n_keep:,}  {ram_now()}", 2)
    
    del mm
    force_gc()
    return out


def load_X_full_clean(path, n_rows, n_feat):
    """Load full X (cho val, te), clean NaN/Inf."""
    arr = np.array(np.memmap(path, dtype="float32", mode="r",
                              shape=(n_rows, n_feat)))
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return arr


def load_y_subsampled(path, n_rows_full, indices):
    """Load y với subsample."""
    mm = np.memmap(path, dtype="float32", mode="r",
                   shape=(n_rows_full,))
    out = np.array(mm[indices])
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    del mm
    return out


def load_y_full(path, n_rows):
    arr = np.array(np.memmap(path, dtype="float32", mode="r",
                              shape=(n_rows,)))
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return arr


def predict_test_batched(model, predict_fn=None):
    """Predict test theo batch, clean NaN từng batch."""
    X_te_mm = np.memmap(X_te_path, dtype="float32", mode="r",
                        shape=(N_TE, N_FEAT))
    out = np.zeros(N_TE, dtype=np.float32)
    n_batches = (N_TE + PREDICT_BATCH - 1) // PREDICT_BATCH
    for bi, i in enumerate(range(0, N_TE, PREDICT_BATCH)):
        j = min(i + PREDICT_BATCH, N_TE)
        batch = np.array(X_te_mm[i:j])
        np.nan_to_num(batch, copy=False, nan=0.0,
                      posinf=0.0, neginf=0.0)
        if predict_fn:
            out[i:j] = predict_fn(model, batch)
        else:
            out[i:j] = model.predict(batch).astype(np.float32)
        del batch
        force_gc()
        if bi % 3 == 0:
            log(f"    pred batch {bi+1}/{n_batches}  {ram_now()}", 2)
    del X_te_mm
    return out


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 4. LIGHTGBM                                                 ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[4] LIGHTGBM")

lgb_vp_path = PRED_DIR / "lgb_vp.npy"
lgb_tp_path = PRED_DIR / "lgb_tp.npy"
lgb_rmse_path = PRED_DIR / "lgb_rmse.txt"

if lgb_vp_path.exists() and lgb_tp_path.exists() and lgb_rmse_path.exists():
    log("LGB preds exist, skip", 1)
    lgb_vp = np.load(lgb_vp_path)
    lgb_tp = np.load(lgb_tp_path)
    lgb_rmse = float(lgb_rmse_path.read_text().strip())
    log(f"  LGB RMSE: {lgb_rmse:.4f}", 1)
else:
    log("Loading X_tr (subsampled, cleaned)...", 1)
    X_tr = load_X_subsampled(X_tr_path, N_TR_FULL, N_FEAT, sub_idx)
    log(f"  X_tr shape: {X_tr.shape}  {ram_now()}", 1)
    
    log("Loading y_tr (subsampled)...", 1)
    y_tr = load_y_subsampled(y_tr_path, N_TR_FULL, sub_idx)
    log(f"  y_tr shape: {y_tr.shape}  {ram_now()}", 1)
    
    log("Loading X_val (cleaned)...", 1)
    X_val = load_X_full_clean(X_val_path, N_VAL, N_FEAT)
    y_val = load_y_full(y_val_path, N_VAL)
    log(f"  Val loaded.  {ram_now()}", 1)
    
    # Verify no NaN/Inf
    assert np.isfinite(X_tr).all(), "X_tr still has NaN/Inf!"
    assert np.isfinite(X_val).all(), "X_val still has NaN/Inf!"
    assert np.isfinite(y_tr).all(), "y_tr has NaN/Inf!"
    assert np.isfinite(y_val).all(), "y_val has NaN/Inf!"
    log("  ✓ No NaN/Inf", 1)
    
    lgb_params = dict(
        objective="tweedie", tweedie_variance_power=1.1,
        metric="rmse", boosting_type="gbdt",
        learning_rate=0.05, num_leaves=63,
        feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1,
        min_child_samples=100,             # tăng từ 50 → 100 chống split lỗi
        min_data_in_leaf=100,
        min_split_gain=0.01,               # ⭐ tránh split với gain quá nhỏ
        reg_alpha=0.1, reg_lambda=1.0,
        seed=42, verbose=-1, max_bin=255,
    )
    
    def train_lgb(params):
        dtr = lgb.Dataset(X_tr, y_tr, feature_name=FCOLS, free_raw_data=True)
        dvl = lgb.Dataset(X_val, y_val, reference=dtr, free_raw_data=True)
        dtr.construct(); dvl.construct()
        m = lgb.train(params, dtr, num_boost_round=N_BOOST,
                      valid_sets=[dvl],
                      callbacks=[lgb.early_stopping(EARLY_STOP),
                                 lgb.log_evaluation(200)])
        return m
    
    log("Training LGB on GPU...", 1)
    try:
        p_gpu = {**lgb_params, "device":"gpu",
                 "gpu_platform_id":0, "gpu_device_id":0}
        lgb_m = train_lgb(p_gpu)
        log("  ✓ LGB on GPU", 1)
    except Exception as e:
        log(f"  GPU failed: {str(e)[:100]}; CPU fallback", 1)
        force_gc()
        lgb_m = train_lgb(lgb_params)
    log(f"  After fit. {ram_now()}", 1)
    
    # Predict val
    lgb_vp = lgb_m.predict(X_val).astype(np.float32)
    np.save(lgb_vp_path, lgb_vp)
    
    # Save y for ensemble step
    np.save(PRED_DIR/"y_val.npy", y_val)
    np.save(PRED_DIR/"y_tr_sample.npy", y_tr[:100000])
    
    # Drop train data
    del X_tr, y_tr, X_val
    force_gc()
    log(f"  After del train. {ram_now()}", 1)
    
    # Predict test
    log("  Predicting test (batched)...", 1)
    lgb_tp = predict_test_batched(lgb_m)
    np.save(lgb_tp_path, lgb_tp)
    
    lgb_rmse = float(np.sqrt(mean_squared_error(y_val, lgb_vp)))
    log(f"  LGB RMSE: {lgb_rmse:.4f}", 1)
    lgb_rmse_path.write_text(f"{lgb_rmse}")
    
    lgb_m.save_model(str(MODELS_DIR/"lgb.txt"))
    fi = pd.DataFrame({"feature":FCOLS,
                        "gain":lgb_m.feature_importance("gain")})\
            .sort_values("gain", ascending=False)
    fi.to_csv(OUTPUT_DIR/"lgb_importance.csv", index=False)
    log(f"  Top 10:\n{fi.head(10)[['feature','gain']].to_string(index=False)}", 1)
    
    del lgb_m, y_val
    force_gc()
    log(f"  After LGB cleanup. {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 5. XGBOOST                                                  ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[5] XGBOOST")

xgb_vp_path = PRED_DIR / "xgb_vp.npy"
xgb_tp_path = PRED_DIR / "xgb_tp.npy"
xgb_rmse_path = PRED_DIR / "xgb_rmse.txt"

if xgb_vp_path.exists() and xgb_tp_path.exists() and xgb_rmse_path.exists():
    log("XGB preds exist, skip", 1)
    xgb_vp = np.load(xgb_vp_path)
    xgb_tp = np.load(xgb_tp_path)
    xgb_rmse = float(xgb_rmse_path.read_text().strip())
    log(f"  XGB RMSE: {xgb_rmse:.4f}", 1)
else:
    log("Loading X_tr (subsampled, cleaned)...", 1)
    X_tr = load_X_subsampled(X_tr_path, N_TR_FULL, N_FEAT, sub_idx)
    y_tr = load_y_subsampled(y_tr_path, N_TR_FULL, sub_idx)
    log(f"  X_tr: {X_tr.shape}  {ram_now()}", 1)
    
    X_val = load_X_full_clean(X_val_path, N_VAL, N_FEAT)
    y_val = load_y_full(y_val_path, N_VAL)
    log(f"  Val loaded.  {ram_now()}", 1)
    
    def train_xgb(device):
        m = xgb.XGBRegressor(
            n_estimators=N_BOOST, learning_rate=0.05,
            max_depth=7, subsample=0.7, colsample_bytree=0.7,
            min_child_weight=100, reg_alpha=0.1, reg_lambda=1.0,
            objective="reg:tweedie", tweedie_variance_power=1.1,
            tree_method="hist", device=device,
            random_state=42, early_stopping_rounds=EARLY_STOP,
            eval_metric="rmse", verbosity=1,
            n_jobs=-1 if device=="cpu" else 1,
            max_bin=256,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)
        return m
    
    log("Training XGB on CUDA...", 1)
    try:
        xgb_m = train_xgb("cuda")
        log("  ✓ XGB on GPU", 1)
    except Exception as e:
        log(f"  GPU failed: {str(e)[:100]}; CPU fallback", 1)
        force_gc()
        xgb_m = train_xgb("cpu")
    
    xgb_vp = xgb_m.predict(X_val).astype(np.float32)
    np.save(xgb_vp_path, xgb_vp)
    
    del X_tr, y_tr, X_val
    force_gc()
    log(f"  After del train. {ram_now()}", 1)
    
    log("  Predicting test (batched)...", 1)
    xgb_tp = predict_test_batched(xgb_m)
    np.save(xgb_tp_path, xgb_tp)
    
    xgb_rmse = float(np.sqrt(mean_squared_error(y_val, xgb_vp)))
    log(f"  XGB RMSE: {xgb_rmse:.4f}", 1)
    xgb_rmse_path.write_text(f"{xgb_rmse}")
    xgb_m.save_model(str(MODELS_DIR/"xgb.json"))
    
    del xgb_m, y_val
    force_gc()
    log(f"  After XGB cleanup. {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 6. ENSEMBLE + METRICS                                       ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[6] ENSEMBLE + METRICS")

# Load y_val
y_val_for_metrics = load_y_full(y_val_path, N_VAL)

# Load y_tr sample
y_tr_sample_path = PRED_DIR / "y_tr_sample.npy"
if y_tr_sample_path.exists():
    y_tr_sample = np.load(y_tr_sample_path)
else:
    y_tr_mm = np.memmap(y_tr_path, dtype="float32", mode="r",
                        shape=(N_TR_FULL,))
    y_tr_sample = np.array(y_tr_mm[:100000])
    del y_tr_mm
    np.nan_to_num(y_tr_sample, copy=False, nan=0.0)

# Ensemble: inverse RMSE weights
sc = np.array([lgb_rmse, xgb_rmse])
w = (1.0/sc); w /= w.sum()
ens_vp = (w[0]*lgb_vp + w[1]*xgb_vp).astype(np.float32)
ens_tp = (w[0]*lgb_tp + w[1]*xgb_tp).astype(np.float32)
model_list = [("LightGBM", lgb_vp), ("XGBoost", xgb_vp), ("Ensemble", ens_vp)]
log(f"Weights: LGB={w[0]:.3f}  XGB={w[1]:.3f}", 1)

def metrics(yt, yp, ynaive=None, name=""):
    yt = np.asarray(yt, dtype=np.float64)
    yp = np.asarray(yp, dtype=np.float64)
    mae  = float(np.mean(np.abs(yt-yp)))
    rmse = float(np.sqrt(np.mean((yt-yp)**2)))
    wape = float(np.sum(np.abs(yt-yp))/(np.sum(np.abs(yt))+1e-9)*100)
    bias = float(np.mean(yp-yt)/(np.mean(yt)+1e-9)*100)
    r2   = float(r2_score(yt, yp))
    if ynaive is not None:
        mase = float(mae/(np.mean(np.abs(np.diff(ynaive.astype(np.float64))))+1e-9))
    else:
        mase = np.nan
    return {"Model":name, "MAE":round(mae,4), "RMSE":round(rmse,4),
            "WAPE%":round(wape,4), "Bias%":round(bias,4),
            "MASE":round(mase,4) if not np.isnan(mase) else "N/A",
            "R2":round(r2,4)}

rows = [metrics(y_val_for_metrics, vp, y_tr_sample, name)
        for name, vp in model_list]
mdf = pd.DataFrame(rows).set_index("Model")
print()
print(mdf.to_string())
mdf.to_csv(OUTPUT_DIR/"metrics.csv")

# Plot
COLORS = ["#4C72B0","#55A868","#C44E52"]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("M5 Validation — Global Model", fontweight="bold")
for a, col, ttl in zip(ax, ["RMSE","MAE","R2"], ["RMSE ↓","MAE ↓","R² ↑"]):
    v = mdf[col].astype(float).values
    bars = a.bar(mdf.index, v, color=COLORS)
    a.set_title(ttl)
    for b, val in zip(bars, v):
        a.text(b.get_x()+b.get_width()/2, val*1.005,
               f"{val:.3f}", ha="center", fontsize=8)
    a.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR/"results.png", dpi=120, bbox_inches="tight")
plt.close(fig)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  STEP 7. SUBMISSION                                               ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[7] SUBMISSION")

test_id_codes = np.load(MMAP_DIR/"test_id_codes.npy")
test_days     = np.load(MMAP_DIR/"test_days.npy")
id_lookup     = np.load(MMAP_DIR/"id_lookup.npy", allow_pickle=True)

pred = np.clip(ens_tp, 0.0, None).astype(np.float32)
test_id_strs = id_lookup[test_id_codes]

sdf = pd.DataFrame({"id":test_id_strs, "day_id":test_days, "pred":pred})\
        .sort_values(["id","day_id"]).reset_index(drop=True)
sdf["fcol"] = "F"+(sdf.groupby("id").cumcount()+1).astype(str)
sub = (sdf.pivot(index="id", columns="fcol", values="pred")
          .reindex(columns=[f"F{i}" for i in range(1, HORIZON+1)])
          .fillna(0).round(4).reset_index())
sub.to_csv(OUTPUT_DIR/"submission.csv", index=False)

fcheck = [f"F{i}" for i in range(1, HORIZON+1)]
n_zero = int((sub[fcheck]==0).all(axis=1).sum())
log(f"  Submission: {sub.shape}", 1)
log(f"  All-zero rows: {n_zero}", 1)
print(sub.head(3).to_string())

del y_val_for_metrics, y_tr_sample
force_gc()

dt = time.time() - t0
print()
print("═"*70)
print(f"  TOTAL TIME: {dt/60:.1f} min")
print(f"  {ram_now()}")
print("═"*70)
print("Done ✓")


══════════════════════════════════════════════════════════════════════
  M5 — TRAIN ONLY v3 (RAM-SAFE)
══════════════════════════════════════════════════════════════════════
  RAM   1.2/33.7GB ( 4.9%) free=32.0GB

══════════════════════════════════════════════════════════════════════
  [1] INSPECT
══════════════════════════════════════════════════════════════════════
  RAM   1.2/33.7GB ( 4.9%) free=32.0GB
    Features: 49
    N_TR_FULL: 56,619,930
    N_VAL:        853,720
    N_TE:         853,720
    Subsample ratio: 0.4

══════════════════════════════════════════════════════════════════════
  [2] BUILD SUBSAMPLE INDICES
══════════════════════════════════════════════════════════════════════
  RAM   1.2/33.7GB ( 4.9%) free=32.0GB
    Built indices: 22,647,972
    → N_TR after subsample: 22,647,972
    → Estimated X_tr size:  4.44 GB

══════════════════════════════════════════════════════════════════════
  [4] LIGHTGBM
══════════════════════════════════════════════════════════════════

[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



    GPU failed: Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_; CPU fallback
Training until validation scores don't improve for 80 rounds
[200]	valid_0's rmse: 1.67391
[400]	valid_0's rmse: 1.66127
[600]	valid_0's rmse: 1.65727
[800]	valid_0's rmse: 1.65496
Early stopping, best iteration is:
[914]	valid_0's rmse: 1.65334
    After fit. RAM   7.5/33.7GB (23.6%) free=25.7GB
    After del train. RAM   2.8/33.7GB ( 9.6%) free=30.4GB
    Predicting test (batched)...
        pred batch 1/3  RAM   2.8/33.7GB ( 9.6%) free=30.4GB
    LGB RMSE: 1.6533
    Top 10:
    feature         gain
zero_streak 3.295167e+08
     ewm_01 1.259260e+08
    rmean_7 5.101894e+07
   rmean_14 2.554712e+07
   rmean_28 1.586657e+07
    rstd_28 9.149513e+06
     ewm_05 8.880983e+06
      lag_1 2.001349e+06
 item_gmean 1.590684e+06
 sell_price 1.303601e+06
    After LGB cleanup. RAM   2.8/33.7GB ( 9.6%) free=30.4GB

══════════════════════════════════════════════════

In [2]:
"""
M5 — STEP 1 v2: RECURSIVE FORECAST (FIXED)
===========================================
Fix: thêm EWM features + handle missing columns gracefully
"""

import os, gc, warnings, time, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import psutil
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "output"
MODELS_DIR  = WORK_DIR / "models"
MMAP_DIR    = WORK_DIR / "mmap"
PARQUET_DIR = WORK_DIR / "parquet"
PRED_DIR    = WORK_DIR / "preds"

HORIZON = 28
LAGS    = [1, 7, 14, 28]

def ram_now():
    m = psutil.virtual_memory()
    return f"RAM {m.used/1e9:.1f}/{m.total/1e9:.1f}GB free={m.available/1e9:.1f}GB"

def log(msg, indent=0):
    print(f"{'  '*indent}{msg}")

def banner(s):
    print("\n" + "═"*70)
    print(f"  {s}")
    print("═"*70)
    print(f"  {ram_now()}")

def force_gc():
    for _ in range(3): gc.collect()

t0 = time.time()
banner("STEP 1 v2: RECURSIVE FORECAST (FIXED)")
# ─────────────────────────────────────────────────────────────────────
# Load feature columns
with open(MMAP_DIR/"feature_cols.json") as f:
    FCOLS = json.load(f)
N_FEAT = len(FCOLS)
log(f"FCOLS ({N_FEAT}): {FCOLS}", 1)

# Detect EWM
HAS_EWM = ("ewm_01" in FCOLS) or ("ewm_05" in FCOLS)
HAS_LAG_364 = "lag_364" in FCOLS
log(f"  HAS_EWM: {HAS_EWM}", 1)
log(f"  HAS_LAG_364: {HAS_LAG_364}", 1)

# Load models
log("Loading models...", 1)
lgb_m = lgb.Booster(model_file=str(MODELS_DIR/"lgb.txt"))
xgb_m = xgb.XGBRegressor()
xgb_m.load_model(str(MODELS_DIR/"xgb.json"))
log("  ✓ LGB + XGB loaded", 1)

lgb_rmse = float((PRED_DIR/"lgb_rmse.txt").read_text().strip())
xgb_rmse = float((PRED_DIR/"xgb_rmse.txt").read_text().strip())
sc = np.array([lgb_rmse, xgb_rmse])
w = (1.0/sc); w /= w.sum()
log(f"  Weights: LGB={w[0]:.3f} XGB={w[1]:.3f}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  Load parquet                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[1] LOAD PARQUET")

parts = []
for fname in ["train", "val", "test"]:
    p = PARQUET_DIR / f"{fname}.parquet"
    df_ = pd.read_parquet(p)
    parts.append(df_)
    log(f"  {fname}: {df_.shape}", 1)
full = pd.concat(parts, ignore_index=True, copy=False)
del parts; force_gc()
full.sort_values(["id_enc","day_id"], inplace=True, kind="stable")
full.reset_index(drop=True, inplace=True)
log(f"  Full: {full.shape}  {ram_now()}", 1)

TRAIN_END = int(full.loc[full["_split"]==0, "day_id"].max())
VAL_END   = int(full.loc[full["_split"]==1, "day_id"].max())
TEST_DAYS = sorted(full.loc[full["_split"]==2, "day_id"].unique().tolist())
log(f"  train_end={TRAIN_END}, val_end={VAL_END}", 1)
log(f"  test days: {TEST_DAYS[0]}..{TEST_DAYS[-1]} ({len(TEST_DAYS)} days)", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  REBUILD STATIC FEATURES                                          ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[2] REBUILD STATIC FEATURES")

# Price
full["sell_price"] = (full.groupby("id_enc", sort=False, observed=True)["sell_price"]
                         .transform(lambda x: x.ffill().bfill())
                         .fillna(0).astype("float32"))
force_gc()

mask_known = full["_split"] < 2
ps = (full.loc[mask_known]
          .groupby(["item_id_enc","store_id_enc"], observed=True)["sell_price"]
          .agg(["mean","min","max"]))
del mask_known; force_gc()
mi = pd.MultiIndex.from_arrays([full["item_id_enc"], full["store_id_enc"]])
full["price_ratio"] = (full["sell_price"]/(mi.map(ps["mean"]).astype("float32")+1e-9)).astype("float32")
pmin = mi.map(ps["min"]).astype("float32")
pmax = mi.map(ps["max"]).astype("float32")
full["price_norm"] = ((full["sell_price"]-pmin)/(pmax-pmin+1e-9)).astype("float32")
del pmin, pmax, ps, mi; force_gc()

# price_diff & price_changed (per id)
full["price_diff"] = (full.groupby("id_enc", sort=False, observed=True)["sell_price"]
                         .diff().fillna(0).astype("float32"))
full["price_changed"] = (full["price_diff"] != 0).astype("int8")
force_gc()
log("  Price features ✓", 1)

# Event
full["has_event"] = ((full["event_type_1_enc"]>0)|(full["event_type_2_enc"]>0)).astype("int8")
for s in ["CA","TX","WI"]:
    full[f"snap_{s}_x_event"] = (full[f"snap_{s}"]*full["has_event"]).astype("int8")
    full[f"wday_x_snap_{s}"]  = (full["wday"]*full[f"snap_{s}"]).astype("int8")
if "day_of_month" in full.columns:
    full["is_payday"] = full["day_of_month"].isin([1,15,28,29,30,31]).astype("int8")
force_gc()

# Aggregate day means
for keys, name in [
    (["store_id_enc","day_id"], "store_day_mean"),
    (["dept_id_enc","day_id"],  "dept_day_mean"),
    (["cat_id_enc","day_id"],   "cat_day_mean"),
    (["day_id"],                "global_day_mean"),
]:
    agg = full.loc[full["_split"]<2].groupby(keys, observed=True)["sales"].mean()
    if len(keys)==1:
        full[name] = full[keys[0]].map(agg).fillna(0).astype("float32")
    else:
        mi2 = pd.MultiIndex.from_arrays([full[k] for k in keys])
        full[name] = mi2.map(agg).fillna(0).astype("float32")
        del mi2
    del agg; force_gc()

im = full.loc[full["_split"]<2].groupby("item_id_enc", observed=True)["sales"].mean()
full["item_gmean"] = full["item_id_enc"].map(im).fillna(0).astype("float32")
del im; force_gc()
log("  Aggregate features ✓", 1)
log(f"  {ram_now()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  COMPUTE LAG/ROLLING/EWM (full features)                          ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[3] COMPUTE DYNAMIC FEATURES")

# Sales_dyn: copy sales, train NaN → 0
full["sales_dyn"] = full["sales"].astype("float32")
m_tr = full["_split"] == 0
full.loc[m_tr & full["sales_dyn"].isna(), "sales_dyn"] = 0.0
del m_tr

def compute_dynamic_features(df, has_ewm=HAS_EWM, has_lag_364=HAS_LAG_364):
    """Tính lag/rolling/ewm dùng sales_dyn."""
    g = df.groupby("id_enc", sort=False, observed=True)["sales_dyn"]
    
    # Lags
    for lag in LAGS:
        df[f"lag_{lag}"] = g.shift(lag).astype("float32")
    if has_lag_364:
        df["lag_364"] = g.shift(364).astype("float32")
    
    # Shift-1 base
    s1 = g.shift(1)
    
    # Rolling means
    for w in [7, 14, 28]:
        df[f"rmean_{w}"] = s1.rolling(w, min_periods=1).mean().astype("float32")
    df["rstd_28"] = s1.rolling(28, min_periods=1).std().fillna(0).astype("float32")
    df["rmax_7"]  = s1.rolling(7, min_periods=1).max().fillna(0).astype("float32")
    
    # EWM
    if has_ewm:
        df["ewm_01"] = s1.ewm(alpha=0.1, min_periods=1).mean().astype("float32")
        df["ewm_05"] = s1.ewm(alpha=0.5, min_periods=1).mean().astype("float32")
    
    df["lag1_diff"] = s1.diff().fillna(0).astype("float32")
    del s1
    
    df["trend_7"] = (df["lag_1"] - df["lag_7"]).fillna(0).astype("float32")
    df["vs_store"] = (df["lag_1"]/(df["store_day_mean"]+1e-9)).clip(-10,10).astype("float32")
    df["vs_item"]  = (df["lag_1"]/(df["item_gmean"]+1e-9)).clip(-10,10).astype("float32")
    
    # zero_streak (vectorized)
    iz = (df["sales_dyn"].fillna(0)==0).astype("int8")
    grp = df["id_enc"]
    # Reset cumsum khi gặp non-zero
    grp_change = (iz != iz.groupby(grp, sort=False).shift().fillna(-1)).cumsum()
    df["zero_streak"] = iz.groupby(grp_change).cumsum().astype("int16").values
    del iz, grp, grp_change
    
    return df

log("Computing dynamic features...", 1)
full = compute_dynamic_features(full)
force_gc()
log(f"  Shape: {full.shape}  {ram_now()}", 1)

# ──── VERIFY all FCOLS present ────
missing = [c for c in FCOLS if c not in full.columns]
if missing:
    log(f"  ⚠️ MISSING COLS: {missing}", 1)
    log("  Adding as zeros...", 1)
    for c in missing:
        full[c] = np.float32(0)

extra = [c for c in full.columns if c not in FCOLS and c not in
         {"sales","sales_dyn","_split","day_id","date"}]
if extra:
    log(f"  Extra cols (ignored): {extra[:5]}{'...' if len(extra)>5 else ''}", 1)

log("  ✓ All FCOLS present", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  CACHE TEST INDICES                                               ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[4] CACHE TEST INDICES")
log("Caching test rows by day...", 1)
test_day_to_idx = {}
for d in TEST_DAYS:
    mask = (full["_split"]==2) & (full["day_id"]==d)
    test_day_to_idx[d] = np.where(mask.to_numpy())[0]
log(f"  Cached {len(test_day_to_idx)} days, "
    f"avg {np.mean([len(v) for v in test_day_to_idx.values()]):.0f} rows/day", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  PREDICT FUNCTION                                                 ║
# ╚════════════════════════════════════════════════════════════════════╝
def predict_ensemble(X_batch):
    lgb_p = lgb_m.predict(X_batch).astype(np.float32)
    xgb_p = xgb_m.predict(X_batch).astype(np.float32)
    np.nan_to_num(lgb_p, copy=False, nan=0.0)
    np.nan_to_num(xgb_p, copy=False, nan=0.0)
    return (w[0]*lgb_p + w[1]*xgb_p).clip(0.0, None)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  RECURSIVE LOOP                                                   ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[5] RECURSIVE FORECAST LOOP")

test_predictions = {}
sales_dyn_col_idx = full.columns.get_loc("sales_dyn")

for step, target_day in enumerate(TEST_DAYS):
    step_t0 = time.time()
    
    test_idx = test_day_to_idx[target_day]
    n_target = len(test_idx)
    
    # Build X — chỉ lấy đúng FCOLS theo thứ tự đã save
    X_target = full.iloc[test_idx][FCOLS].to_numpy(dtype=np.float32, copy=True)
    np.nan_to_num(X_target, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Predict
    preds = predict_ensemble(X_target)
    test_predictions[target_day] = preds
    
    # Fill predictions
    full.iloc[test_idx, sales_dyn_col_idx] = preds
    
    del X_target
    
    # Recompute features cho ngày kế tiếp (nếu còn)
    if step < HORIZON - 1:
        full = compute_dynamic_features(full)
        # Re-fill missing cols nếu cần (safety)
        for c in missing:
            if c not in full.columns:
                full[c] = np.float32(0)
    
    elapsed = time.time() - step_t0
    log(f"  Day {target_day} ({step+1:>2}/28): n={n_target:,} "
        f"mean={preds.mean():.3f} max={preds.max():.1f} "
        f"t={elapsed:.1f}s | {ram_now()}", 1)
    force_gc()


# ╔════════════════════════════════════════════════════════════════════╗
# ║  BUILD SUBMISSION                                                 ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[6] BUILD SUBMISSION")

all_ids, all_days, all_preds = [], [], []
for d in TEST_DAYS:
    idx = test_day_to_idx[d]
    all_ids.append(full.iloc[idx]["id_enc"].to_numpy())
    all_days.append(np.full(len(idx), d, dtype=np.int32))
    all_preds.append(test_predictions[d])

all_ids   = np.concatenate(all_ids)
all_days  = np.concatenate(all_days)
all_preds = np.concatenate(all_preds)

id_lookup = np.load(MMAP_DIR/"id_lookup.npy", allow_pickle=True)
test_id_strs = id_lookup[all_ids]

sdf = pd.DataFrame({
    "id":     test_id_strs,
    "day_id": all_days,
    "pred":   all_preds.clip(0.0, None).round(4),
}).sort_values(["id","day_id"]).reset_index(drop=True)

sdf["fcol"] = "F" + (sdf.groupby("id").cumcount()+1).astype(str)
sub = (sdf.pivot(index="id", columns="fcol", values="pred")
          .reindex(columns=[f"F{i}" for i in range(1, HORIZON+1)])
          .fillna(0).round(4).reset_index())

sub.to_csv(OUTPUT_DIR/"submission_recursive.csv", index=False)

fcheck = [f"F{i}" for i in range(1, HORIZON+1)]
n_zero = int((sub[fcheck]==0).all(axis=1).sum())
log(f"  Shape: {sub.shape}", 1)
log(f"  All-zero rows: {n_zero} (vs original ~812)", 1)
log(f"  Mean pred: {sub[fcheck].values.mean():.4f}", 1)
log(f"  Saved → submission_recursive.csv", 1)
print(sub.head(3).to_string())

np.save(PRED_DIR/"recursive_preds.npy", all_preds)
np.save(PRED_DIR/"recursive_ids.npy",   all_ids)
np.save(PRED_DIR/"recursive_days.npy",  all_days)

dt = time.time() - t0
print()
print("═"*70)
print(f"  STEP 1 v2 DONE in {dt/60:.1f} min")
print(f"  {ram_now()}")
print("═"*70)


══════════════════════════════════════════════════════════════════════
  STEP 1 v2: RECURSIVE FORECAST (FIXED)
══════════════════════════════════════════════════════════════════════
  RAM 3.0/33.7GB free=30.2GB
  FCOLS (49): ['sell_price', 'wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI', 'id_enc', 'item_id_enc', 'dept_id_enc', 'cat_id_enc', 'store_id_enc', 'state_id_enc', 'quarter', 'is_weekend', 'day_of_month', 'event_type_1_enc', 'event_type_2_enc', 'has_event', 'snap_CA_x_event', 'wday_x_snap_CA', 'snap_TX_x_event', 'wday_x_snap_TX', 'snap_WI_x_event', 'wday_x_snap_WI', 'is_payday', 'price_ratio', 'price_norm', 'item_gmean', 'store_day_mean', 'dept_day_mean', 'cat_day_mean', 'global_day_mean', 'zero_streak', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rmean_7', 'rmean_14', 'rmean_28', 'rstd_28', 'rmax_7', 'ewm_01', 'ewm_05', 'lag1_diff', 'trend_7', 'vs_store', 'vs_item']
    HAS_EWM: True
    HAS_LAG_364: False
  Loading models...
    ✓ LGB + XGB loaded
    Weights: LGB=0.501 X

In [3]:
"""
M5 — STEP 2: MULTI-LOSS MODELS
================================
Thêm 2 models với loss function khác:
  - LGB với Poisson loss (intermittent demand thân thiện)
  - LGB với RMSE loss (sạch sẽ, baseline)
  - Tăng tweedie_variance_power lên 1.5 cho LGB cũ (sparse-friendly)

Ensemble: 4 models (LGB-tweedie, LGB-poisson, LGB-rmse, XGB)

PEAK RAM: ~12 GB
TIME: ~60 phút
"""

import os, gc, warnings, time, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import psutil
from pathlib import Path

import lightgbm as lgb
from sklearn.metrics import mean_squared_error

# ─────────────────────────────────────────────────────────────────────
WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "output"
MODELS_DIR  = WORK_DIR / "models"
MMAP_DIR    = WORK_DIR / "mmap"
PRED_DIR    = WORK_DIR / "preds"

HORIZON       = 28
N_BOOST       = 1200
EARLY_STOP    = 80
PREDICT_BATCH = 300_000
LOAD_CHUNK    = 2_000_000

def ram_now():
    m = psutil.virtual_memory()
    return f"RAM {m.used/1e9:.1f}/{m.total/1e9:.1f}GB free={m.available/1e9:.1f}GB"

def log(msg, indent=0):
    print(f"{'  '*indent}{msg}")

def banner(s):
    print("\n" + "═"*70)
    print(f"  {s}")
    print("═"*70)
    print(f"  {ram_now()}")

def force_gc():
    for _ in range(3): gc.collect()

t0 = time.time()
banner("STEP 2: MULTI-LOSS MODELS")

# Load shared
with open(MMAP_DIR/"feature_cols.json") as f:
    FCOLS = json.load(f)
N_FEAT = len(FCOLS)

X_tr_path  = MMAP_DIR / "X_tr.dat"
X_val_path = MMAP_DIR / "X_val.dat"
X_te_path  = MMAP_DIR / "X_te.dat"
y_tr_path  = MMAP_DIR / "y_tr.dat"
y_val_path = MMAP_DIR / "y_val.dat"

def infer_n(path, n_feat=1):
    return path.stat().st_size // (n_feat * 4)

N_TR_FULL = infer_n(X_tr_path, N_FEAT)
N_VAL     = infer_n(X_val_path, N_FEAT)
N_TE      = infer_n(X_te_path,  N_FEAT)

sub_idx = np.load(MMAP_DIR/"train_sub_idx.npy")
N_TR = len(sub_idx)
log(f"N_TR={N_TR:,} N_VAL={N_VAL:,} N_TE={N_TE:,}", 1)


def load_X_subsampled(path, n_full, n_feat, idx):
    out = np.empty((len(idx), n_feat), dtype=np.float32)
    mm = np.memmap(path, dtype="float32", mode="r", shape=(n_full, n_feat))
    pos = 0
    for cs in range(0, len(idx), LOAD_CHUNK):
        ce = min(cs+LOAD_CHUNK, len(idx))
        block = np.asarray(mm[idx[cs:ce]])
        np.nan_to_num(block, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        out[pos:pos+(ce-cs)] = block
        pos += (ce-cs)
        del block
        if (cs//LOAD_CHUNK) % 3 == 0:
            force_gc()
    del mm; force_gc()
    return out

def load_X_full_clean(path, n, n_feat):
    arr = np.array(np.memmap(path, dtype="float32", mode="r", shape=(n, n_feat)))
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return arr

def load_y_subsampled(path, n_full, idx):
    mm = np.memmap(path, dtype="float32", mode="r", shape=(n_full,))
    out = np.array(mm[idx])
    np.nan_to_num(out, copy=False, nan=0.0)
    del mm
    return out

def load_y_full(path, n):
    arr = np.array(np.memmap(path, dtype="float32", mode="r", shape=(n,)))
    np.nan_to_num(arr, copy=False, nan=0.0)
    return arr

def predict_test_batched(model):
    X_te_mm = np.memmap(X_te_path, dtype="float32", mode="r", shape=(N_TE, N_FEAT))
    out = np.zeros(N_TE, dtype=np.float32)
    for i in range(0, N_TE, PREDICT_BATCH):
        j = min(i+PREDICT_BATCH, N_TE)
        batch = np.array(X_te_mm[i:j])
        np.nan_to_num(batch, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        out[i:j] = model.predict(batch).astype(np.float32)
        del batch; force_gc()
    del X_te_mm
    return out


# Common LGB base params
BASE_PARAMS = dict(
    metric="rmse", boosting_type="gbdt",
    learning_rate=0.05, num_leaves=63,
    feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1,
    min_child_samples=100, min_data_in_leaf=100,
    min_split_gain=0.01,
    reg_alpha=0.1, reg_lambda=1.0,
    seed=42, verbose=-1, max_bin=255,
)

# ─────────────────────────────────────────────────────────────────────
# MODEL 1: LGB-TWEEDIE-1.5 (higher variance_power for intermittent)
# ─────────────────────────────────────────────────────────────────────
banner("[1] LGB-TWEEDIE-1.5")
vp_path = PRED_DIR/"lgb_tw15_vp.npy"
tp_path = PRED_DIR/"lgb_tw15_tp.npy"
rm_path = PRED_DIR/"lgb_tw15_rmse.txt"

if vp_path.exists() and tp_path.exists() and rm_path.exists():
    log("Skip (preds exist)", 1)
    tw15_vp = np.load(vp_path)
    tw15_tp = np.load(tp_path)
    tw15_rmse = float(rm_path.read_text().strip())
else:
    log("Loading data...", 1)
    X_tr = load_X_subsampled(X_tr_path, N_TR_FULL, N_FEAT, sub_idx)
    y_tr = load_y_subsampled(y_tr_path, N_TR_FULL, sub_idx)
    X_val = load_X_full_clean(X_val_path, N_VAL, N_FEAT)
    y_val = load_y_full(y_val_path, N_VAL)
    log(f"  Loaded. {ram_now()}", 1)

    params = {**BASE_PARAMS,
              "objective":"tweedie",
              "tweedie_variance_power":1.5}
    
    def train(p):
        dtr = lgb.Dataset(X_tr, y_tr, feature_name=FCOLS, free_raw_data=True)
        dvl = lgb.Dataset(X_val, y_val, reference=dtr, free_raw_data=True)
        dtr.construct(); dvl.construct()
        return lgb.train(p, dtr, num_boost_round=N_BOOST,
                          valid_sets=[dvl],
                          callbacks=[lgb.early_stopping(EARLY_STOP),
                                     lgb.log_evaluation(200)])
    try:
        m = train({**params, "device":"gpu", "gpu_platform_id":0, "gpu_device_id":0})
        log("  ✓ GPU", 1)
    except Exception as e:
        log(f"  GPU failed: {str(e)[:80]}; CPU", 1)
        m = train(params)
    
    tw15_vp = m.predict(X_val).astype(np.float32)
    np.save(vp_path, tw15_vp)
    del X_tr, y_tr, X_val; force_gc()
    log(f"  After del. {ram_now()}", 1)
    
    log("  Predict test...", 1)
    tw15_tp = predict_test_batched(m)
    np.save(tp_path, tw15_tp)
    
    tw15_rmse = float(np.sqrt(mean_squared_error(y_val, tw15_vp)))
    log(f"  RMSE: {tw15_rmse:.4f}", 1)
    rm_path.write_text(f"{tw15_rmse}")
    m.save_model(str(MODELS_DIR/"lgb_tw15.txt"))
    del m, y_val; force_gc()


# ─────────────────────────────────────────────────────────────────────
# MODEL 2: LGB-POISSON
# ─────────────────────────────────────────────────────────────────────
banner("[2] LGB-POISSON")
vp_path = PRED_DIR/"lgb_pois_vp.npy"
tp_path = PRED_DIR/"lgb_pois_tp.npy"
rm_path = PRED_DIR/"lgb_pois_rmse.txt"

if vp_path.exists() and tp_path.exists() and rm_path.exists():
    log("Skip (preds exist)", 1)
    pois_vp = np.load(vp_path)
    pois_tp = np.load(tp_path)
    pois_rmse = float(rm_path.read_text().strip())
else:
    log("Loading data...", 1)
    X_tr = load_X_subsampled(X_tr_path, N_TR_FULL, N_FEAT, sub_idx)
    y_tr = load_y_subsampled(y_tr_path, N_TR_FULL, sub_idx)
    X_val = load_X_full_clean(X_val_path, N_VAL, N_FEAT)
    y_val = load_y_full(y_val_path, N_VAL)
    log(f"  Loaded. {ram_now()}", 1)

    # Poisson cần y >= 0 integer-like
    y_tr_p  = np.maximum(y_tr, 0).round().astype(np.float32)
    y_val_p = np.maximum(y_val, 0).round().astype(np.float32)
    
    params = {**BASE_PARAMS,
              "objective":"poisson",
              "poisson_max_delta_step":0.7}
    
    def train(p):
        dtr = lgb.Dataset(X_tr, y_tr_p, feature_name=FCOLS, free_raw_data=True)
        dvl = lgb.Dataset(X_val, y_val_p, reference=dtr, free_raw_data=True)
        dtr.construct(); dvl.construct()
        return lgb.train(p, dtr, num_boost_round=N_BOOST,
                          valid_sets=[dvl],
                          callbacks=[lgb.early_stopping(EARLY_STOP),
                                     lgb.log_evaluation(200)])
    try:
        m = train({**params, "device":"gpu", "gpu_platform_id":0, "gpu_device_id":0})
        log("  ✓ GPU", 1)
    except Exception as e:
        log(f"  GPU failed: {str(e)[:80]}; CPU", 1)
        m = train(params)
    
    pois_vp = m.predict(X_val).astype(np.float32)
    np.save(vp_path, pois_vp)
    del X_tr, y_tr, y_tr_p, X_val; force_gc()
    log(f"  After del. {ram_now()}", 1)
    
    log("  Predict test...", 1)
    pois_tp = predict_test_batched(m)
    np.save(tp_path, pois_tp)
    
    pois_rmse = float(np.sqrt(mean_squared_error(y_val, pois_vp)))
    log(f"  RMSE: {pois_rmse:.4f}", 1)
    rm_path.write_text(f"{pois_rmse}")
    m.save_model(str(MODELS_DIR/"lgb_pois.txt"))
    del m, y_val, y_val_p; force_gc()


# ─────────────────────────────────────────────────────────────────────
# MODEL 3: LGB-RMSE (regression baseline)
# ─────────────────────────────────────────────────────────────────────
banner("[3] LGB-RMSE")
vp_path = PRED_DIR/"lgb_rmse_vp.npy"
tp_path = PRED_DIR/"lgb_rmse_tp.npy"
rm_path = PRED_DIR/"lgb_rmse_rmse.txt"

if vp_path.exists() and tp_path.exists() and rm_path.exists():
    log("Skip (preds exist)", 1)
    rmse_vp = np.load(vp_path)
    rmse_tp = np.load(tp_path)
    rmse_rmse = float(rm_path.read_text().strip())
else:
    log("Loading data...", 1)
    X_tr = load_X_subsampled(X_tr_path, N_TR_FULL, N_FEAT, sub_idx)
    y_tr = load_y_subsampled(y_tr_path, N_TR_FULL, sub_idx)
    X_val = load_X_full_clean(X_val_path, N_VAL, N_FEAT)
    y_val = load_y_full(y_val_path, N_VAL)
    log(f"  Loaded. {ram_now()}", 1)

    params = {**BASE_PARAMS, "objective":"regression"}
    
    def train(p):
        dtr = lgb.Dataset(X_tr, y_tr, feature_name=FCOLS, free_raw_data=True)
        dvl = lgb.Dataset(X_val, y_val, reference=dtr, free_raw_data=True)
        dtr.construct(); dvl.construct()
        return lgb.train(p, dtr, num_boost_round=N_BOOST,
                          valid_sets=[dvl],
                          callbacks=[lgb.early_stopping(EARLY_STOP),
                                     lgb.log_evaluation(200)])
    try:
        m = train({**params, "device":"gpu", "gpu_platform_id":0, "gpu_device_id":0})
        log("  ✓ GPU", 1)
    except Exception as e:
        log(f"  GPU failed: {str(e)[:80]}; CPU", 1)
        m = train(params)
    
    rmse_vp = m.predict(X_val).astype(np.float32)
    np.save(vp_path, rmse_vp)
    del X_tr, y_tr, X_val; force_gc()
    
    log("  Predict test...", 1)
    rmse_tp = predict_test_batched(m)
    np.save(tp_path, rmse_tp)
    
    rmse_rmse = float(np.sqrt(mean_squared_error(y_val, rmse_vp)))
    log(f"  RMSE: {rmse_rmse:.4f}", 1)
    rm_path.write_text(f"{rmse_rmse}")
    m.save_model(str(MODELS_DIR/"lgb_rmse.txt"))
    del m, y_val; force_gc()


# ─────────────────────────────────────────────────────────────────────
# ENSEMBLE 5 MODELS: original LGB+XGB + 3 new
# ─────────────────────────────────────────────────────────────────────
banner("[4] ENSEMBLE 5 MODELS")

# Load tất cả preds
lgb_vp = np.load(PRED_DIR/"lgb_vp.npy")
lgb_tp = np.load(PRED_DIR/"lgb_tp.npy")
xgb_vp = np.load(PRED_DIR/"xgb_vp.npy")
xgb_tp = np.load(PRED_DIR/"xgb_tp.npy")

lgb_rmse_orig = float((PRED_DIR/"lgb_rmse.txt").read_text().strip())
xgb_rmse_orig = float((PRED_DIR/"xgb_rmse.txt").read_text().strip())

# Load recursive predictions nếu có
rec_path = PRED_DIR/"recursive_preds.npy"
has_recursive = rec_path.exists()
if has_recursive:
    log("  Found recursive preds from Step 1!", 1)
    rec_preds = np.load(rec_path)
    # rec_preds đã ở format giống lgb_tp
    use_rec_tp = rec_preds
else:
    use_rec_tp = None

models = {
    "LGB-tweedie-1.1": (lgb_vp, lgb_tp, lgb_rmse_orig),
    "LGB-tweedie-1.5": (tw15_vp, tw15_tp, tw15_rmse),
    "LGB-poisson":     (pois_vp, pois_tp, pois_rmse),
    "LGB-rmse":        (rmse_vp, rmse_tp, rmse_rmse),
    "XGB":             (xgb_vp, xgb_tp, xgb_rmse_orig),
}

y_val_full = load_y_full(y_val_path, N_VAL)

# Inverse-RMSE weights
sc = np.array([v[2] for v in models.values()])
w = (1.0/sc); w /= w.sum()
names = list(models.keys())

log("Weights:", 1)
for n, wv, s in zip(names, w, sc):
    log(f"  {n:<20} w={wv:.3f}  RMSE={s:.4f}", 2)

# Ensemble preds
val_stack = np.stack([m[0] for m in models.values()])
test_stack = np.stack([m[1] for m in models.values()])

ens_vp = (w[:, None] * val_stack).sum(axis=0).astype(np.float32)
ens_tp = (w[:, None] * test_stack).sum(axis=0).astype(np.float32)

# Nếu có recursive preds, blend với ensemble (50/50)
if has_recursive:
    log("  Blending recursive 50% + ensemble 50%...", 1)
    ens_tp = (0.5*ens_tp + 0.5*use_rec_tp).astype(np.float32)

# Metrics
def metrics(yt, yp, name):
    yt = np.asarray(yt, dtype=np.float64)
    yp = np.asarray(yp, dtype=np.float64)
    return {
        "Model": name,
        "MAE":   round(float(np.mean(np.abs(yt-yp))), 4),
        "RMSE":  round(float(np.sqrt(np.mean((yt-yp)**2))), 4),
        "WAPE%": round(float(np.sum(np.abs(yt-yp))/(np.sum(np.abs(yt))+1e-9)*100), 4),
        "Bias%": round(float(np.mean(yp-yt)/(np.mean(yt)+1e-9)*100), 4),
    }

rows = [metrics(y_val_full, v[0], n) for n, v in models.items()]
rows.append(metrics(y_val_full, ens_vp, "ENSEMBLE-5"))
mdf = pd.DataFrame(rows).set_index("Model")
print()
print(mdf.to_string())
mdf.to_csv(OUTPUT_DIR/"metrics_step2.csv")


# ─────────────────────────────────────────────────────────────────────
# SUBMISSION
# ─────────────────────────────────────────────────────────────────────
banner("[5] SUBMISSION")
test_id_codes = np.load(MMAP_DIR/"test_id_codes.npy")
test_days     = np.load(MMAP_DIR/"test_days.npy")
id_lookup     = np.load(MMAP_DIR/"id_lookup.npy", allow_pickle=True)

pred = np.clip(ens_tp, 0.0, None).astype(np.float32)
test_id_strs = id_lookup[test_id_codes]

sdf = pd.DataFrame({"id":test_id_strs, "day_id":test_days, "pred":pred})\
        .sort_values(["id","day_id"]).reset_index(drop=True)
sdf["fcol"] = "F"+(sdf.groupby("id").cumcount()+1).astype(str)
sub = (sdf.pivot(index="id", columns="fcol", values="pred")
          .reindex(columns=[f"F{i}" for i in range(1, HORIZON+1)])
          .fillna(0).round(4).reset_index())
sub.to_csv(OUTPUT_DIR/"submission_step2.csv", index=False)

fcheck = [f"F{i}" for i in range(1, HORIZON+1)]
n_zero = int((sub[fcheck]==0).all(axis=1).sum())
log(f"  Shape: {sub.shape}, all-zero rows: {n_zero}", 1)
log(f"  Saved → submission_step2.csv", 1)

# Save preds cho step 3
np.save(PRED_DIR/"step2_ens_tp.npy", ens_tp)
np.save(PRED_DIR/"step2_ens_vp.npy", ens_vp)

dt = time.time() - t0
print()
print("═"*70)
print(f"  STEP 2 DONE in {dt/60:.1f} min")
print(f"  {ram_now()}")
print("═"*70)


══════════════════════════════════════════════════════════════════════
  STEP 2: MULTI-LOSS MODELS
══════════════════════════════════════════════════════════════════════
  RAM 11.7/33.7GB free=21.5GB
  N_TR=22,647,972 N_VAL=853,720 N_TE=853,720

══════════════════════════════════════════════════════════════════════
  [1] LGB-TWEEDIE-1.5
══════════════════════════════════════════════════════════════════════
  RAM 11.7/33.7GB free=21.5GB
  Loading data...
    Loaded. RAM 16.4/33.7GB free=16.8GB
Training until validation scores don't improve for 80 rounds


[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



    GPU failed: Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src; CPU
Training until validation scores don't improve for 80 rounds
[200]	valid_0's rmse: 1.67822
[400]	valid_0's rmse: 1.66239
[600]	valid_0's rmse: 1.6566
[800]	valid_0's rmse: 1.65431
[1000]	valid_0's rmse: 1.65232
[1200]	valid_0's rmse: 1.65101
Did not meet early stopping. Best iteration is:
[1160]	valid_0's rmse: 1.6509
    After del. RAM 13.6/33.7GB free=19.6GB
    Predict test...
    RMSE: 1.6509

══════════════════════════════════════════════════════════════════════
  [2] LGB-POISSON
══════════════════════════════════════════════════════════════════════
  RAM 13.5/33.7GB free=19.6GB
  Loading data...
    Loaded. RAM 18.0/33.7GB free=15.2GB
Training until validation scores don't improve for 80 rounds
[200]	valid_0's rmse: 1.72322


[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



    GPU failed: Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src; CPU
Training until validation scores don't improve for 80 rounds
[200]	valid_0's rmse: 1.72424
[400]	valid_0's rmse: 1.70343
[600]	valid_0's rmse: 1.6927
[800]	valid_0's rmse: 1.68577
[1000]	valid_0's rmse: 1.6809
[1200]	valid_0's rmse: 1.67742
Did not meet early stopping. Best iteration is:
[1200]	valid_0's rmse: 1.67742
    After del. RAM 14.5/33.7GB free=18.7GB
    Predict test...
    RMSE: 1.6774

══════════════════════════════════════════════════════════════════════
  [3] LGB-RMSE
══════════════════════════════════════════════════════════════════════
  RAM 14.4/33.7GB free=18.7GB
  Loading data...
    Loaded. RAM 18.9/33.7GB free=14.3GB


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 80 rounds
[200]	valid_0's rmse: 1.69002
[400]	valid_0's rmse: 1.67975
[600]	valid_0's rmse: 1.6765
[800]	valid_0's rmse: 1.67467
[1000]	valid_0's rmse: 1.67394
[1200]	valid_0's rmse: 1.67266
Did not meet early stopping. Best iteration is:
[1200]	valid_0's rmse: 1.67266
    ✓ GPU
    Predict test...
    RMSE: 1.6727

══════════════════════════════════════════════════════════════════════
  [4] ENSEMBLE 5 MODELS
══════════════════════════════════════════════════════════════════════
  RAM 14.5/33.7GB free=18.7GB
    Found recursive preds from Step 1!
  Weights:
      LGB-tweedie-1.1      w=0.201  RMSE=1.6533
      LGB-tweedie-1.5      w=0.201  RMSE=1.6509
      LGB-poisson          w=0.198  RMSE=1.6774
      LGB-rmse             w=0.199  RMSE=1.6727
      XGB                  w=0.201  RMSE=1.6571
    Blending recursive 50% + ensemble 50%...

                    MAE    RMSE    WAPE%   Bias%
Model                                           
L

In [4]:
"""
M5 — STEP 3: HIERARCHICAL RECONCILIATION
==========================================
M5 hierarchy có 12 levels:
  Level 1:  Total (1 series)
  Level 2:  State (3 series: CA, TX, WI)
  Level 3:  Store (10 series)
  Level 4:  Category (3 series: FOODS, HOBBIES, HOUSEHOLD)
  Level 5:  Dept (7 series)
  Level 6:  State × Category
  Level 7:  State × Dept
  Level 8:  Store × Category
  Level 9:  Store × Dept
  Level 10: Item
  Level 11: Item × State
  Level 12: Item × Store (bottom, 30490 series)

Phương pháp đơn giản & hiệu quả:
  - BOTTOM-UP: predict ở bottom (item×store) → aggregate lên các levels
  - MIDDLE-OUT: predict middle levels (state×dept), redistribute xuống bottom
  - MinT (Minimum Trace): tối ưu nhưng phức tạp

Ta dùng kết hợp:
  1. Predict bottom (đã có từ step 1 hoặc 2)
  2. Predict total/state/store levels riêng (đơn giản hơn vì ít series)
  3. Reconcile: bottom_reconciled = bottom_pred × (total_pred / bottom_sum)
  
  Đây là "TOP-DOWN proportional adjustment" — đơn giản nhưng work tốt cho M5.

PEAK RAM: ~6 GB
TIME: ~20 phút
"""

import os, gc, warnings, time, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import psutil
from pathlib import Path

import lightgbm as lgb
from sklearn.metrics import mean_squared_error

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "output"
MODELS_DIR  = WORK_DIR / "models"
MMAP_DIR    = WORK_DIR / "mmap"
PARQUET_DIR = WORK_DIR / "parquet"
PRED_DIR    = WORK_DIR / "preds"

HORIZON = 28

def ram_now():
    m = psutil.virtual_memory()
    return f"RAM {m.used/1e9:.1f}/{m.total/1e9:.1f}GB free={m.available/1e9:.1f}GB"

def log(msg, indent=0):
    print(f"{'  '*indent}{msg}")

def banner(s):
    print("\n" + "═"*70)
    print(f"  {s}")
    print("═"*70)
    print(f"  {ram_now()}")

def force_gc():
    for _ in range(3): gc.collect()

t0 = time.time()
banner("STEP 3: HIERARCHICAL RECONCILIATION")


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [1] LOAD bottom-level preds (từ step 1 hoặc 2)                   ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[1] LOAD BOTTOM PREDS")

# Ưu tiên step 2 ensemble, fallback step 1 recursive, fallback original
if (PRED_DIR/"step2_ens_tp.npy").exists():
    bottom_tp = np.load(PRED_DIR/"step2_ens_tp.npy")
    source = "step2_ensemble"
elif (PRED_DIR/"recursive_preds.npy").exists():
    bottom_tp = np.load(PRED_DIR/"recursive_preds.npy")
    source = "step1_recursive"
else:
    # Build from original
    lgb_tp = np.load(PRED_DIR/"lgb_tp.npy")
    xgb_tp = np.load(PRED_DIR/"xgb_tp.npy")
    lgb_rmse = float((PRED_DIR/"lgb_rmse.txt").read_text().strip())
    xgb_rmse = float((PRED_DIR/"xgb_rmse.txt").read_text().strip())
    sc = np.array([lgb_rmse, xgb_rmse])
    w = (1.0/sc); w /= w.sum()
    bottom_tp = (w[0]*lgb_tp + w[1]*xgb_tp).astype(np.float32)
    source = "original"
log(f"Source: {source}, shape: {bottom_tp.shape}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [2] LOAD parquet để biết hierarchy structure                     ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[2] LOAD HIERARCHY")

parts = []
for fname in ["train", "val", "test"]:
    p = PARQUET_DIR / f"{fname}.parquet"
    df_ = pd.read_parquet(p)
    # Chỉ giữ cột cần
    keep = ["id_enc","item_id_enc","dept_id_enc","cat_id_enc",
            "store_id_enc","state_id_enc","day_id","sales","_split"]
    parts.append(df_[[c for c in keep if c in df_.columns]])
    del df_
full = pd.concat(parts, ignore_index=True, copy=False)
del parts; force_gc()
log(f"Full: {full.shape}  {ram_now()}", 1)

# Test info
test_id_codes = np.load(MMAP_DIR/"test_id_codes.npy")
test_days     = np.load(MMAP_DIR/"test_days.npy")
log(f"Test rows: {len(test_id_codes):,}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [3] BUILD HIERARCHY LOOKUPS                                      ║
# ║  Map từ id_enc → state, store, dept, cat                          ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[3] BUILD HIERARCHY LOOKUPS")

# Lấy 1 row per id_enc (hierarchy info constant per series)
hier = (full[["id_enc","item_id_enc","dept_id_enc",
              "cat_id_enc","store_id_enc","state_id_enc"]]
         .drop_duplicates(subset=["id_enc"])
         .set_index("id_enc")
         .sort_index())
log(f"Hierarchy: {hier.shape}", 1)
# Verify: 30490 unique id_enc
N_SERIES = len(hier)
log(f"  N_series (bottom level): {N_SERIES}", 1)
log(f"  N_states:  {hier['state_id_enc'].nunique()}", 1)
log(f"  N_stores:  {hier['store_id_enc'].nunique()}", 1)
log(f"  N_depts:   {hier['dept_id_enc'].nunique()}", 1)
log(f"  N_cats:    {hier['cat_id_enc'].nunique()}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [4] BUILD AGGREGATE TIME SERIES (state/store/dept)               ║
# ║  Để train models đơn giản trên top levels                         ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[4] AGGREGATE TIME SERIES")

# Merge hierarchy vào full
full = full.merge(hier.reset_index()[["id_enc","state_id_enc","store_id_enc",
                                       "dept_id_enc","cat_id_enc"]]
                       .rename(columns={"state_id_enc":"st", "store_id_enc":"sr",
                                       "dept_id_enc":"dp", "cat_id_enc":"ct"}),
                  on="id_enc", how="left")

# State-level aggregate (train + val only, has sales)
log("Aggregating state-level...", 1)
state_ts = (full.loc[full["_split"]<2]
                 .groupby(["state_id_enc","day_id"], observed=True)["sales"]
                 .sum().reset_index())
log(f"  State series: {state_ts['state_id_enc'].nunique()}, "
    f"days: {state_ts['day_id'].nunique()}", 1)

# Store-level
log("Aggregating store-level...", 1)
store_ts = (full.loc[full["_split"]<2]
                 .groupby(["store_id_enc","day_id"], observed=True)["sales"]
                 .sum().reset_index())
log(f"  Store series: {store_ts['store_id_enc'].nunique()}", 1)

# Total (global)
log("Aggregating global...", 1)
total_ts = (full.loc[full["_split"]<2]
                 .groupby("day_id", observed=True)["sales"]
                 .sum().reset_index())


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [5] FORECAST TOP LEVELS bằng simple moving average               ║
# ║  Với 10 stores × 28 days và 3 states × 28 days, model đơn giản đủ ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[5] FORECAST TOP LEVELS")

# Simple method: predict bằng exp-weighted moving average (alpha=0.3)
# + seasonal adjustment (cùng wday trong tuần)

# Lấy 28 ngày gần nhất + same-wday-history
TEST_DAYS_LIST = sorted(np.unique(test_days).tolist())
log(f"Forecasting {len(TEST_DAYS_LIST)} days × top levels", 1)

# Tính wday cho test days từ parquet
test_day_to_wday = (full[full["day_id"].isin(TEST_DAYS_LIST)]
                     [["day_id","st"]].drop_duplicates())  # dummy
# Lấy wday từ parquet (giả sử có cột wday — không có thì compute)

# Đơn giản: predict bằng mean của 28 ngày gần nhất
def forecast_level(ts_df, group_col):
    """
    ts_df: DataFrame với cols [group_col, day_id, sales]
    Return: dict {group_id: array of 28 predictions}
    """
    preds = {}
    for g, sub in ts_df.groupby(group_col, observed=True):
        sub = sub.sort_values("day_id")
        sales = sub["sales"].values.astype(np.float32)
        # Method 1: EWMA của 28 ngày cuối
        recent = sales[-56:] if len(sales) >= 56 else sales
        # Tính avg theo wday (cycle 7)
        wday_means = np.zeros(7, dtype=np.float32)
        for w in range(7):
            vals = recent[w::7]
            wday_means[w] = vals.mean() if len(vals) > 0 else recent.mean()
        # Project 28 days
        forecast = np.tile(wday_means, 4)[:HORIZON].astype(np.float32)
        preds[g] = forecast
    return preds

log("  Forecasting state level...", 1)
state_preds = forecast_level(state_ts, "state_id_enc")
state_total = sum(state_preds.values())
log(f"    Total state forecast (day 1): {state_total[0]:.0f}", 2)

log("  Forecasting store level...", 1)
store_preds = forecast_level(store_ts, "store_id_enc")
store_total = sum(store_preds.values())
log(f"    Total store forecast (day 1): {store_total[0]:.0f}", 2)

# Total: tổng từ state hoặc store level (consistency check)
log(f"  Total state vs store: {state_total[:3]} vs {store_total[:3]}", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [6] BUILD BOTTOM-UP AGGREGATES từ predictions                    ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[6] BOTTOM-UP AGGREGATES")

# Tạo DataFrame test predictions với hierarchy
test_df = pd.DataFrame({
    "id_enc": test_id_codes,
    "day_id": test_days,
    "pred":   bottom_tp,
})
test_df = test_df.merge(hier.reset_index()[["id_enc","state_id_enc","store_id_enc"]],
                        on="id_enc", how="left")

# Aggregate bottom predictions to state and store levels
bu_state = (test_df.groupby(["state_id_enc","day_id"], observed=True)["pred"]
                   .sum().reset_index())
bu_store = (test_df.groupby(["store_id_enc","day_id"], observed=True)["pred"]
                   .sum().reset_index())

log("Bottom-up state preds (vs top-down):", 1)
for s in sorted(bu_state["state_id_enc"].unique()):
    sub = bu_state[bu_state["state_id_enc"]==s].sort_values("day_id")
    bu_day1 = sub["pred"].iloc[0]
    td_day1 = state_preds[s][0]
    log(f"  state {s}: bottom-up={bu_day1:.1f}, top-down={td_day1:.1f}, "
        f"ratio={bu_day1/(td_day1+1e-9):.3f}", 2)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [7] RECONCILE — PROPORTIONAL ADJUSTMENT                          ║
# ║  Per state, per day: scale bottom preds để khớp top-down forecast ║
# ║  Blend với weight 0.7 bottom-up + 0.3 reconciled                  ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[7] RECONCILE")

# Build adjustment ratio per (state, day): top_down / bottom_up
adj_state = {}
for s in sorted(test_df["state_id_enc"].unique()):
    bu = bu_state[bu_state["state_id_enc"]==s].sort_values("day_id")["pred"].values
    td = state_preds[s]
    ratio = td / (bu + 1e-9)
    # Clip extreme ratios
    ratio = np.clip(ratio, 0.5, 2.0)
    adj_state[s] = ratio
    log(f"  state {s} adjustment range: {ratio.min():.3f}~{ratio.max():.3f}", 1)

# Apply to bottom predictions
test_df = test_df.sort_values(["id_enc","day_id"]).reset_index(drop=True)
day_idx_map = {d: i for i, d in enumerate(TEST_DAYS_LIST)}

reconciled = np.zeros_like(bottom_tp)
for i, row in enumerate(test_df.itertuples(index=False)):
    s = row.state_id_enc
    di = day_idx_map[row.day_id]
    reconciled[i] = row.pred * adj_state[s][di]

# Blend: 70% bottom-up, 30% reconciled (conservative)
BLEND_W = 0.3
reconciled_final = ((1-BLEND_W)*test_df["pred"].values + BLEND_W*reconciled).astype(np.float32)
log(f"  Blend: {(1-BLEND_W)*100:.0f}% bottom + {BLEND_W*100:.0f}% reconciled", 1)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  [8] BUILD SUBMISSION                                             ║
# ╚════════════════════════════════════════════════════════════════════╝
banner("[8] SUBMISSION")

id_lookup = np.load(MMAP_DIR/"id_lookup.npy", allow_pickle=True)
test_id_strs = id_lookup[test_df["id_enc"].values]

sdf = pd.DataFrame({
    "id":     test_id_strs,
    "day_id": test_df["day_id"].values,
    "pred":   np.clip(reconciled_final, 0.0, None).round(4),
}).sort_values(["id","day_id"]).reset_index(drop=True)

sdf["fcol"] = "F" + (sdf.groupby("id").cumcount()+1).astype(str)
sub = (sdf.pivot(index="id", columns="fcol", values="pred")
          .reindex(columns=[f"F{i}" for i in range(1, HORIZON+1)])
          .fillna(0).round(4).reset_index())

sub.to_csv(OUTPUT_DIR/"submission_reconciled.csv", index=False)

fcheck = [f"F{i}" for i in range(1, HORIZON+1)]
n_zero = int((sub[fcheck]==0).all(axis=1).sum())
log(f"  Shape: {sub.shape}, all-zero rows: {n_zero}", 1)
log(f"  Mean pred: {sub[fcheck].values.mean():.4f}", 1)
log(f"  Saved → submission_reconciled.csv", 1)

# Save for next step
np.save(PRED_DIR/"step3_reconciled.npy", reconciled_final)

dt = time.time() - t0
print()
print("═"*70)
print(f"  STEP 3 DONE in {dt/60:.1f} min")
print(f"  {ram_now()}")
print("═"*70)


══════════════════════════════════════════════════════════════════════
  STEP 3: HIERARCHICAL RECONCILIATION
══════════════════════════════════════════════════════════════════════
  RAM 14.4/33.7GB free=18.8GB

══════════════════════════════════════════════════════════════════════
  [1] LOAD BOTTOM PREDS
══════════════════════════════════════════════════════════════════════
  RAM 14.4/33.7GB free=18.8GB
  Source: step2_ensemble, shape: (853720,)

══════════════════════════════════════════════════════════════════════
  [2] LOAD HIERARCHY
══════════════════════════════════════════════════════════════════════
  RAM 14.4/33.7GB free=18.8GB
  Full: (59181090, 9)  RAM 6.6/33.7GB free=26.5GB
  Test rows: 853,720

══════════════════════════════════════════════════════════════════════
  [3] BUILD HIERARCHY LOOKUPS
══════════════════════════════════════════════════════════════════════
  RAM 6.6/33.7GB free=26.5GB
  Hierarchy: (30490, 5)
    N_series (bottom level): 30490
    N_states:  3
    N_